# Experimento VQE — versão 20.16

## Causalidade estrutural + geometria variacional + soma sobre caminhos

Esta versão contém integralmente o experimento **20.15** e acrescenta uma nova etapa para investigar matematicamente por que apenas um subconjunto dos 30 parâmetros controla a solução.

A organização é deliberadamente cumulativa:

1. reproduz a varredura e o teste dos **23 θ aparentemente insensíveis**;
2. preserva os testes causais do 20.15, deslocando os **7 blocos sensíveis**;
3. calcula a geometria variacional local por **QGT/Fubini–Study**;
4. mede gradiente e, principalmente, **curvatura energética** de cada θ;
5. calcula a influência de cada θ sobre a amplitude e a probabilidade do bitstring ótimo;
6. acompanha o estado **bloco a bloco**;
7. reconstrói uma **soma discreta sobre caminhos** com interferência coerente;
8. mede a dimensão efetiva da variedade e o comprimento Fubini–Study de uma trajetória parametrizada.

> **Regra de interpretação:** este notebook não chama a trajetória do VQE de “caminho de mínima ação”. Em um VQE estático não existe, por padrão, uma coordenada temporal física. O princípio variacional é usado aqui por meio das quantidades geométricas e tangentes bem definidas do ansatz. A soma sobre caminhos é calculada explicitamente como uma decomposição discreta das amplitudes do circuito.


## Como interpretar o circuito antes dos testes

O circuito começa aplicando portas `X` em $k$ qubits. Isso prepara **um estado-base inicial com peso de Hamming $k$**; não significa que a solução ótima já foi inserida no circuito.

Depois, os blocos parametrizados redistribuem a amplitude entre estados que mantêm a cardinalidade:

- `CY`: bloco lógico de **dois qubits**, decomposto com uma rotação controlada `CRY`;
- `CCY`: bloco lógico de **três qubits**, decomposto com `RY` e `CCX`;
- `RY`: operação primitiva de um qubit;
- `CX`: operação primitiva de dois qubits;
- `CCX`: operação primitiva de três qubits.

Portanto, não é correto dizer que toda porta é simplesmente uma junção de dois spins. O Hamiltoniano do portfólio é diagonal e contém termos de um e dois corpos, `Z` e `ZZ`, enquanto o **ansatz** usa blocos de dois e três qubits para navegar no subespaço de Dicke.

O índice $j$ em $\theta_j$ não deve ser fornecido isoladamente ao Transformer como significado físico. O objeto transferível é a descrição estrutural:

$$
(\text{tipo de bloco},\; \text{qubits/ativos},\; \text{distância},\;
\text{posição},\; \text{período},\; \text{termos do Hamiltoniano tocados}).
$$


### Célula 1 — Importações, parâmetros e pastas do experimento

**Em termos simples:** esta célula configura somente o que é necessário para selecionar 100 vetores e executar as varreduras individuais.

**O que é configurado:**

- os parâmetros financeiros $q$, $r_f$ e a cardinalidade $k=4$;
- a máscara dos 10% melhores;
- a seleção final de exatamente 100 vetores completos;
- os índices $\theta_j$ testados;
- 201 pontos regulares por período, além do valor original inserido exatamente;
- checkpoints por par `(vetor, theta)`, permitindo retomar uma execução interrompida;
- um diretório novo de saída para as tabelas e figuras corrigidas.

A grade usa o período estrutural de cada porta: `RY` em $[0,2\pi]$ e `CRY` em $[0,4\pi]$. Uma auditoria posterior verifica se, para o observável de probabilidade, os dois ciclos de uma `CRY` são de fato diferentes.

A versão 20.7 procura primeiro seus próprios checkpoints e, quando eles ainda não existem, reutiliza automaticamente os checkpoints da versão 20.6. Assim, a campanha de aproximadamente 180 mil avaliações não precisa ser refeita apenas para corrigir os resumos e gráficos.


In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
import ast
import hashlib
import json
import math
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import spearmanr

NOTEBOOK_VERSION = "20.16-structural-action-pathsum"
RANDOM_SEED = 42

# Parâmetros do problema financeiro.
Q_VALUE = 0.5
RISK_FREE = 0.0475
TARGET_K = 4
ENERGY_ATOL = 1e-8

# ------------------------------------------------------------
# ÚNICO CAMINHO DE ENTRADA
# Altere somente esta linha quando o merge.pkl estiver em outro local.
# ------------------------------------------------------------
MERGE_PKL = Path(r"C:\Users\Marlon_Kelly\Downloads\merge.pkl")

# A máscara conserva aproximadamente os 10% melhores vetores salvos.
TOP_FRACTION = 0.10

# Reavalia mais candidatos do que o número final para escolher 100 vetores
# usando as métricas exatas reconstruídas neste notebook.
MAX_EXACT_REEVALUATION = 300
N_AUDIT_VECTORS = 100
N_ANCHORS = 1

# Parâmetros varridos individualmente em cada um dos 100 vetores.
ACTIVE_THETA_INDICES = [2, 14, 17, 19, 22, 25, 27]
DETAILED_THETA_INDICES = [17, 2, 14, 19, 22, 25, 27, 3, 24]

# 201 pontos regulares por período, mais o ponto original quando necessário.
# Isso controla o custo: 100 vetores x 9 thetas x aproximadamente 202 pontos.
SWEEP_POINTS_PER_PERIOD = 201
COMMON_PHASE_POINTS = 201

# Critério apenas para informar se uma curva é numericamente plana.
SWEEP_FLAT_ABS_TOL = 1e-10
SWEEP_FLAT_REL_TOL = 1e-8
PERIODICITY_ATOL = 1e-9
PERIODICITY_RTOL = 1e-7

# Checkpoints novos desta versão podem ser reutilizados para retomar a campanha.
REUSE_SWEEP_CHECKPOINTS = True

# Saídas corrigidas da versão 20.7.
OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_16_structural_action_pathsum"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DISTRIBUTION_DIR = OUTPUT_ROOT / "distributions"

# Checkpoints produzidos pela versão 20.6 são matematicamente compatíveis,
# pois o motor de varredura não foi alterado; somente a análise foi corrigida.
LEGACY_OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_11_single_vector_symmetric_sweep"
LEGACY_CHECKPOINT_DIR = LEGACY_OUTPUT_ROOT / "checkpoints"

for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, DISTRIBUTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "merge_pkl": str(MERGE_PKL),
    "target_k": TARGET_K,
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "top_fraction": TOP_FRACTION,
    "n_audit_vectors": N_AUDIT_VECTORS,
    "n_sweep_vectors": N_ANCHORS,
    "max_exact_reevaluation": MAX_EXACT_REEVALUATION,
    "active_theta_indices": ACTIVE_THETA_INDICES,
    "detailed_theta_indices": DETAILED_THETA_INDICES,
    "sweep_points_per_period": SWEEP_POINTS_PER_PERIOD,
    "reuse_sweep_checkpoints": REUSE_SWEEP_CHECKPOINTS,
    "legacy_checkpoint_dir": str(LEGACY_CHECKPOINT_DIR),
    "optimizer_used": False,
    "cobyla_calls": 0,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("Saídas:", OUTPUT_ROOT.resolve())
print("Checkpoints legados procurados em:", LEGACY_CHECKPOINT_DIR.resolve())

# Parte I — carregar e auditar somente o `merge.pkl`

O carregamento não procura nomes alternativos em vários diretórios. Existe um único caminho configurado em `MERGE_PKL`.

A célula seguinte aceita `DataFrame`, lista de dicionários ou dicionário serializado, mas não reconstrói nem modifica o banco original.


### Célula 2 — Carregamento controlado do banco `merge.pkl`

**Em termos simples:** esta célula abre o único arquivo de entrada e o transforma em um `DataFrame` de trabalho.

Ela verifica se o caminho existe, aceita três formatos serializados (`DataFrame`, lista de registros ou dicionário) e interrompe a execução caso o arquivo esteja vazio ou tenha um tipo inesperado.

**Saída principal:** `merge_df`, que contém o banco original carregado em memória. O arquivo em disco não é modificado.


In [ ]:
# ============================================================
# 2. CARREGAMENTO ÚNICO DO merge.pkl
# ============================================================

# Resolve "~", converte para caminho absoluto e verifica o arquivo antes da leitura.
merge_path = MERGE_PKL.expanduser().resolve()
if not merge_path.is_file():
    raise FileNotFoundError(
        "merge.pkl não encontrado. Caminho configurado: "
        f"{merge_path}"
    )

# O pickle é lido uma única vez. As conversões seguintes ocorrem apenas em memória.
loaded_object = pd.read_pickle(merge_path)

# Padroniza diferentes formatos serializados para um único DataFrame.
if isinstance(loaded_object, pd.DataFrame):
    merge_df = loaded_object.copy()
elif isinstance(loaded_object, list):
    merge_df = pd.DataFrame(loaded_object)
elif isinstance(loaded_object, dict):
    merge_df = pd.DataFrame(loaded_object)
else:
    raise TypeError(
        "O merge.pkl deve conter DataFrame, lista de registros ou dicionário; "
        f"tipo encontrado: {type(loaded_object)}"
    )

if merge_df.empty:
    raise ValueError("O merge.pkl foi carregado, mas está vazio.")

print("Arquivo:", merge_path)
print("Shape:", merge_df.shape)
print("Colunas:", merge_df.columns.tolist())
display(merge_df.head())


### Célula 3 — Identificação das colunas e conversão dos dados

**Em termos simples:** bancos gerados em versões diferentes podem usar nomes diferentes para a mesma informação. Esta célula cria um mapa de aliases e identifica qual coluna representa retorno, covariância, vetor de parâmetros, energia, probabilidade e bitstring.

Também são definidas funções para converter conteúdos salvos como texto em objetos numéricos:

- `parse_tickers`: recupera os nomes dos ativos;
- `parse_vector`: transforma retornos e vetores $\theta$ em arrays;
- `parse_matrix`: reconstrói a matriz de covariância;
- `normalize_bitstring`: padroniza bitstrings para uma sequência de zeros e uns.

**Importante:** essa normalização ocorre apenas na cópia em memória. O `merge.pkl` original permanece intacto.


In [ ]:
# ============================================================
# 3. NORMALIZAÇÃO DO ESQUEMA SEM ALTERAR O ARQUIVO ORIGINAL
# ============================================================

# Cada chave representa um conceito do experimento; a lista contém nomes de
# coluna aceitos para esse mesmo conceito em versões diferentes do banco.
COLUMN_ALIASES = {
    "tickers": ["tickers", "assets", "asset_names"],
    "assets_return": ["assets_return", "assets_returns", "expected_returns", "mu"],
    "covariance": ["covariance", "covariance_matrix", "sigma"],
    "best_parameters": ["best_parameters", "theta", "theta_final"],
    "initial_point": ["initial_point", "initial_theta", "theta_initial"],
    "objective": ["objective_function_value", "energy", "final_energy"],
    "best_objective": ["best_objective_function_value", "exact_energy", "optimal_energy"],
    "p_best": [
        "p_exact_eval",
        "probability_best_answer",
        "probability_best_answer_shots",
        "p_best",
        "prob_best",
    ],
    "gap": ["gap_exact_eval", "energy_gap", "gap"],
    "dominant_bitstring": [
        "most_frequent_bitstring",
        "most_frequen_bitstring",
        "best_answer",
        "dominant_bitstring",
    ],
    "counts": ["counts", "measurement_counts"],
    "status": ["status"],
}


def first_existing_column(frame, aliases, required=False):
    """Retorna o primeiro alias realmente presente no DataFrame."""
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"Nenhuma das colunas obrigatórias foi encontrada: {aliases}")
    return None


# Resultado final do mapeamento: conceito lógico -> nome real no merge.pkl.
RESOLVED_COLUMNS = {
    key: first_existing_column(
        merge_df,
        aliases,
        required=key in {"tickers", "assets_return", "covariance", "best_parameters"},
    )
    for key, aliases in COLUMN_ALIASES.items()
}


def parse_serialized(value):
    """Converte texto serializado em lista, dicionário ou array quando possível."""
    if isinstance(value, (np.ndarray, list, tuple, dict, pd.Series, pd.Index, pd.DataFrame)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                pass
        cleaned = text.strip("[]()")
        arr = np.fromstring(cleaned.replace(",", " "), sep=" ")
        if arr.size:
            return arr
    return value


def parse_tickers(value):
    """Padroniza a lista de tickers e rejeita nomes vazios."""
    parsed = parse_serialized(value)
    if isinstance(parsed, str):
        items = [item.strip() for item in parsed.replace(";", ",").split(",")]
    elif isinstance(parsed, dict):
        items = list(parsed.keys())
    else:
        items = list(parsed)
    tickers = [str(item).strip().strip("'\"") for item in items]
    if not tickers or any(not item for item in tickers):
        raise ValueError(f"Tickers inválidos: {value}")
    return tickers


def parse_vector(value, tickers=None):
    """Converte um vetor salvo em texto, Series, dicionário ou lista para NumPy."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.Series):
        if tickers is not None and set(tickers).issubset(set(parsed.index.astype(str))):
            return parsed.reindex(tickers).to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        if tickers is not None and set(tickers).issubset(set(map(str, parsed.keys()))):
            return np.asarray([parsed[ticker] for ticker in tickers], dtype=float)
        return np.asarray(list(parsed.values()), dtype=float)
    return np.asarray(parsed, dtype=float).reshape(-1)


def parse_matrix(value, tickers=None):
    """Reconstrói uma matriz numérica e preserva a ordem dos tickers quando possível."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.DataFrame):
        if tickers is not None:
            return parsed.loc[tickers, tickers].to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        frame = pd.DataFrame(parsed)
        if tickers is not None and set(tickers).issubset(frame.index) and set(tickers).issubset(frame.columns):
            return frame.loc[tickers, tickers].to_numpy(dtype=float)
        return frame.to_numpy(dtype=float)
    array = np.asarray(parsed, dtype=float)
    if array.ndim == 1:
        n = int(round(np.sqrt(array.size)))
        if n * n != array.size:
            raise ValueError("Covariância unidimensional não forma uma matriz quadrada.")
        array = array.reshape(n, n)
    return array


def normalize_bitstring(value):
    """Remove prefixos e espaços, retornando apenas bitstrings binários válidos."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).replace(" ", "").replace("'", "").replace('"', "")
    if text.startswith("0b"):
        text = text[2:]
    return text if set(text).issubset({"0", "1"}) else None


print(json.dumps(RESOLVED_COLUMNS, indent=2, ensure_ascii=False))


### Célula 4 — Reconstrução do problema financeiro e auditoria do banco

**Em termos simples:** a primeira linha válida do banco é usada para recuperar os ativos, o vetor de retornos $\mu$ e a matriz de covariância $\Sigma$.

A covariância é explicitamente simetrizada:

$$
\Sigma_{\mathrm{sim}}=\frac{\Sigma+\Sigma^\mathsf{T}}{2}.
$$

Depois, uma amostra de até 200 linhas recebe uma impressão digital (`hash`). Se aparecer mais de um hash, o banco contém mais de um problema/Hamiltoniano e a execução é interrompida.

A célula também estima a cardinalidade pelos bitstrings salvos e cria `problem_summary_df`, com retorno, variância e conexão de risco de cada ativo.


In [ ]:
# ============================================================
# 4. EXTRAIR O HAMILTONIANO E AUDITAR CONSISTÊNCIA DO BANCO
# ============================================================

# Usa a primeira linha com um vetor theta válido como referência do problema.
first_valid_index = merge_df[
    merge_df[RESOLVED_COLUMNS["best_parameters"]].notna()
].index[0]
first_row = merge_df.loc[first_valid_index]

tickers = parse_tickers(first_row[RESOLVED_COLUMNS["tickers"]])
mu = parse_vector(first_row[RESOLVED_COLUMNS["assets_return"]], tickers=tickers)
sigma = parse_matrix(first_row[RESOLVED_COLUMNS["covariance"]], tickers=tickers)
# Corrige pequenas assimetrias numéricas sem alterar a parte simétrica do risco.
sigma = 0.5 * (sigma + sigma.T)

N_ASSETS = len(tickers)
if mu.shape != (N_ASSETS,):
    raise ValueError(f"Retornos com shape {mu.shape}; esperado {(N_ASSETS,)}.")
if sigma.shape != (N_ASSETS, N_ASSETS):
    raise ValueError(
        f"Covariância com shape {sigma.shape}; esperado {(N_ASSETS, N_ASSETS)}."
    )
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância possuem valores não finitos.")

# A cardinalidade é inferida do bitstring salvo quando possível.
bit_col = RESOLVED_COLUMNS["dominant_bitstring"]
if bit_col is not None:
    saved_bits = merge_df[bit_col].map(normalize_bitstring).dropna()
    inferred_weights = saved_bits.map(lambda value: value.count("1"))
    inferred_k = int(inferred_weights.mode().iloc[0]) if not inferred_weights.empty else TARGET_K
else:
    inferred_k = TARGET_K

if inferred_k != TARGET_K:
    warnings.warn(
        f"A cardinalidade modal inferida foi k={inferred_k}; "
        f"o experimento está configurado para k={TARGET_K}."
    )

# Confirma que uma amostra do merge representa o mesmo problema.
def problem_fingerprint(row):
    """Cria um hash a partir de tickers, retornos e covariância de uma linha."""
    row_tickers = parse_tickers(row[RESOLVED_COLUMNS["tickers"]])
    row_mu = parse_vector(row[RESOLVED_COLUMNS["assets_return"]], tickers=row_tickers)
    row_sigma = parse_matrix(row[RESOLVED_COLUMNS["covariance"]], tickers=row_tickers)
    payload = (
        "|".join(row_tickers).encode("utf-8")
        + np.asarray(row_mu, dtype=np.float64).tobytes()
        + np.asarray(row_sigma, dtype=np.float64).tobytes()
    )
    return hashlib.sha256(payload).hexdigest()[:16]

# Uma amostra aleatória é suficiente para detectar mistura evidente de problemas,
# sem reler e converter necessariamente todas as linhas do banco.
sample_size = min(200, len(merge_df))
sampled_rows = merge_df.sample(sample_size, random_state=RANDOM_SEED)
fingerprints = sampled_rows.apply(problem_fingerprint, axis=1)
if fingerprints.nunique() != 1:
    raise RuntimeError(
        "O merge.pkl contém mais de um Hamiltoniano na amostra auditada. "
        "Este notebook 20.6 executa um problema por vez; filtre o merge antes de continuar."
    )

DATA_HASH = fingerprints.iloc[0]
min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())

# Tabela por ativo usada posteriormente como descrição estrutural do problema.
problem_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

print("n =", N_ASSETS, "| k =", TARGET_K)
print("tickers =", tickers)
print("problem_hash =", DATA_HASH)
print("menor autovalor da covariância =", f"{min_cov_eigenvalue:.3e}")
display(problem_summary_df)


# Parte II — referência clássica e rigidez dos pares

## O que esta parte representa

Antes de analisar o circuito quântico, o notebook resolve exatamente o problema clássico. Como existem 10 ativos e o portfólio deve selecionar 4, o número total de soluções válidas é

$$
\binom{10}{4}=210.
$$

Isso significa que é possível avaliar todos os 210 portfólios e conhecer, sem aproximação:

- a energia mínima exata;
- todos os bitstrings ótimos;
- a distância energética entre decisões concorrentes;
- quais pares de ativos possuem decisões mais rígidas.

## Mínimos condicionados de cada par

Para cada par de ativos $(i,j)$, fixamos os valores de decisão $x_i=a$ e $x_j=b$, com $a,b\in\{0,1\}$. Em seguida, procuramos o melhor portfólio que respeita essas duas decisões e continua selecionando exatamente $k$ ativos:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_{\ell}x_{\ell}=k}}
E(x).
$$

Assim, cada par possui quatro energias condicionadas:

$$
E_{ij}^{00},\qquad
E_{ij}^{01},\qquad
E_{ij}^{10},\qquad
E_{ij}^{11}.
$$

Esses valores permitem medir quanto custa trocar a decisão de um ativo, dos dois ativos e quão separado está o melhor estado do par em relação ao segundo melhor. Mais adiante, essas métricas serão ligadas aos blocos quânticos que atuam sobre os mesmos qubits.


### Célula 5 — Solução clássica exata e rigidez dos pares de ativos

**Em termos simples:** para 10 ativos escolhendo exatamente 4, todos os portfólios válidos podem ser enumerados:

$$
N_{\mathrm{portfólios}}=\binom{10}{4}=210.
$$

A função objetivo avaliada para cada bitstring é

$$
E(x)=q\,x^\mathsf{T}\Sigma x-(1-q)\,\mu^\mathsf{T}x+r_f,
$$

com a restrição $\sum_i x_i=k$.

Para cada par de ativos $(i,j)$ e cada estado $a,b\in\{0,1\}$, é calculado o melhor portfólio condicionado:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_\ell x_\ell=k}}
E(x).
$$

Esses quatro mínimos permitem medir:

- `G_ij`: separação entre o melhor e o segundo melhor estado condicionado do par;
- gaps de trocar apenas $i$, apenas $j$ ou os dois;
- não aditividade da troca conjunta.

**Saídas principais:** `enumeration_df`, `asset_decision_df`, `pair_gap_df`, `exact_energy` e os bitstrings ótimos.


In [ ]:
# ============================================================
# 5. ENUMERAÇÃO CLÁSSICA EXATA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary):
    """Calcula E(x)=q*x.T*Sigma*x-(1-q)*mu.T*x+r_f para um portfólio binário."""
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma @ x
        - (1.0 - Q_VALUE) * mu @ x
        + RISK_FREE
    )


def bitstring_asset_order(x):
    """Converte o vetor binário para a ordem natural dos ativos."""
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def enumerate_portfolios(k_value=TARGET_K):
    """Enumera exatamente todas as combinações de k ativos entre N_ASSETS."""
    rows = []
    for selected_indices in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_asset_order(x)
        rows.append({
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(tickers[index] for index in selected_indices),
            "objective": portfolio_objective(x),
        })
    return pd.DataFrame(rows).sort_values(
        ["objective", "bitstring_asset_order"]
    ).reset_index(drop=True)


# Para n=10 e k=4, esta tabela possui C(10,4)=210 linhas.
enumeration_df = enumerate_portfolios(TARGET_K)
exact_energy = float(enumeration_df.iloc[0]["objective"])
optimal_mask = np.isclose(
    enumeration_df["objective"].to_numpy(dtype=float),
    exact_energy,
    atol=ENERGY_ATOL,
    rtol=0.0,
)
optimal_df = enumeration_df.loc[optimal_mask].copy()
exact_asset_bitstrings = sorted(optimal_df["bitstring_asset_order"].unique())
exact_qiskit_bitstrings = sorted(optimal_df["bitstring_qiskit_order"].unique())
exact_x = np.asarray([int(value) for value in exact_asset_bitstrings[0]], dtype=int)
energy_span = float(enumeration_df["objective"].max() - exact_energy)


def build_asset_decision_margins():
    """Mede o custo mínimo de inverter a decisão de cada ativo no ótimo clássico."""
    selected = np.flatnonzero(exact_x == 1)
    excluded = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(N_ASSETS):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append(portfolio_objective(trial))
        else:
            for removed in selected:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append(portfolio_objective(trial))
        margin = max(min(alternatives) - exact_energy, 0.0)
        rows.append({
            "asset_index": asset_index,
            "ticker": tickers[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": float(margin),
            "decision_margin_relative": float(margin / max(energy_span, ENERGY_ATOL)),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False)


asset_decision_df = build_asset_decision_margins()


def conditional_pair_best(i, j, state):
    """Retorna o melhor portfólio sob x_i=a e x_j=b para um estado ab."""
    a, b = int(state[0]), int(state[1])
    subset = enumeration_df.loc[
        enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
    ]
    if subset.empty:
        raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
    return subset.iloc[0]


# Para cada par, calculamos E_00, E_01, E_10 e E_11 e derivamos os gaps.
pair_rows = []
for i, j in combinations(range(N_ASSETS), 2):
    states = {state: conditional_pair_best(i, j, state) for state in PAIR_STATES}
    ordered = sorted(PAIR_STATES, key=lambda state: (states[state]["objective"], state))
    best_state, second_state = ordered[:2]
    global_state = f"{exact_x[i]}{exact_x[j]}"
    a_star, b_star = map(int, global_state)
    flip_i = f"{1-a_star}{b_star}"
    flip_j = f"{a_star}{1-b_star}"
    flip_both = f"{1-a_star}{1-b_star}"
    delta_i = max(float(states[flip_i]["objective"] - exact_energy), 0.0)
    delta_j = max(float(states[flip_j]["objective"] - exact_energy), 0.0)
    delta_both = max(float(states[flip_both]["objective"] - exact_energy), 0.0)
    # G_ij mede quão rigidamente o melhor estado condicionado do par se separa do segundo.
    gij = max(float(states[second_state]["objective"] - states[best_state]["objective"]), 0.0)
    pair_rows.append({
        "asset_index_i": i,
        "asset_index_j": j,
        "asset_i": tickers[i],
        "asset_j": tickers[j],
        "pair": f"{tickers[i]}/{tickers[j]}",
        "global_pair_state": global_state,
        "G_ij": gij,
        "G_ij_relative": gij / max(energy_span, ENERGY_ATOL),
        "single_flip_i_gap": delta_i,
        "single_flip_j_gap": delta_j,
        "joint_flip_gap": delta_both,
        "joint_flip_nonadditivity": delta_both - delta_i - delta_j,
        **{f"E_{state}": float(states[state]["objective"]) for state in PAIR_STATES},
    })

pair_gap_df = pd.DataFrame(pair_rows).sort_values("G_ij", ascending=False).reset_index(drop=True)

print("Portfólios válidos:", len(enumeration_df))
print("Energia exata:", exact_energy)
print("Bitstring ótimo — ordem dos ativos:", exact_asset_bitstrings)
print("Bitstring ótimo — ordem Qiskit:", exact_qiskit_bitstrings)
print("Ativos selecionados:", optimal_df.iloc[0]["selected_assets"])
display(asset_decision_df)
display(pair_gap_df.head(15))


# Parte III — reconstrução exata do Hamiltoniano e do ansatz do modelo 20.1

A construção abaixo foi separada do gerador de banco, mas preserva a mesma lógica do notebook 20.1:

- mesmo QUBO/Ising;
- mesmo estado inicial de peso $k$;
- mesmos blocos `CY` e `CCY`;
- mesma ordenação rastreável dos parâmetros;
- mesmo `ANSATZ_SEED`.

O notebook interrompe a execução caso o vetor salvo tenha dimensão incompatível ou caso a auditoria estrutural falhe.


### Célula 6 — Dependências quânticas sem importação de otimizadores

**Em termos simples:** esta célula carrega somente as classes necessárias para montar o QUBO, converter para Ising, construir o circuito e calcular o `Statevector`.

Nenhum método como COBYLA, SPSA ou outro otimizador variacional é importado. Isso garante que as células seguintes apenas atribuam valores de $\theta$ e avaliem diretamente o circuito.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS — SEM OTIMIZADOR
# ============================================================

# Dependências para formular o problema binário e construir o circuito.
# Não há importação de classes de otimização variacional.
from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

print("Dependências quânticas carregadas. Nenhum otimizador foi importado.")


### Célula 7 — Construção do QUBO e do Hamiltoniano de Ising

**Em termos simples:** o problema financeiro clássico é escrito com variáveis binárias $x_i\in\{0,1\}$ e a restrição de selecionar exatamente $k$ ativos:

$$
\sum_i x_i=k.
$$

O modelo é convertido para QUBO e depois para um Hamiltoniano de Ising:

$$
H=\sum_i h_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j+\text{constante}.
$$

A célula exige que o Hamiltoniano seja diagonal, isto é, sem termos `X` ou `Y`. Em seguida, compara a energia mínima do Ising com a energia obtida pela enumeração dos 210 portfólios. Se elas não coincidirem, a execução para.

**Saídas principais:** `ising`, `ising_offset` e `hamiltonian_terms_df`.


In [ ]:
# ============================================================
# 7. QUBO E HAMILTONIANO DE ISING
# ============================================================


def build_docplex_ising():
    """Constrói o modelo binário, converte para QUBO/Ising e audita a energia exata."""
    model = Model(name=f"portfolio_n{N_ASSETS}_k{TARGET_K}")
    variables = np.array(
        [model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)],
        dtype=object,
    )
    # Termo quadrático de risco x^T Sigma x.
    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS)
        for column in range(N_ASSETS)
    )
    # Termo linear de retorno esperado mu^T x.
    return_expression = model.sum(
        float(mu[index]) * variables[index]
        for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    # Restrição de cardinalidade: exatamente TARGET_K variáveis devem valer 1.
    model.add_constraint(
        model.sum(variables.tolist()) == int(TARGET_K),
        ctname="budget",
    )

    # Docplex -> QuadraticProgram -> QUBO -> operador Ising e deslocamento constante.
    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    # O problema de portfólio deve gerar apenas termos diagonais I, Z e ZZ.
    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    # Auditoria independente: a menor energia Ising deve coincidir com a enumeração.
    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy_ising = float(np.real(exact_result.eigenvalue + offset))
    if not np.isclose(exact_energy_ising, exact_energy, atol=1e-10, rtol=0.0):
        raise RuntimeError(
            "Energia Ising e enumeração clássica não coincidem: "
            f"{exact_energy_ising} vs {exact_energy}"
        )

    terms_df = pd.DataFrame({
        "pauli_label": labels,
        "coefficient": np.real(np.asarray(ising.coeffs)).astype(float),
        "body_order": [label.count("Z") for label in labels],
    })
    return model, quadratic_program, qubo, ising, float(offset), terms_df


model, quadratic_program, qubo, ising, ising_offset, hamiltonian_terms_df = build_docplex_ising()
print("Termos Ising:", len(hamiltonian_terms_df))
display(hamiltonian_terms_df.sort_values(["body_order", "pauli_label"]))


### Célula 8 — Construção rastreável do ansatz de Dicke

**Em termos simples:** esta célula reproduz o circuito parametrizado usado no experimento 20.1 e registra a origem estrutural de cada parâmetro.

- `CY_parameterized` cria um bloco lógico de dois qubits cuja operação parametrizada primitiva é `CRY`;
- `CCY_parameterized` cria um bloco lógico de três qubits usando `RY` e `CCX`;
- as portas `X` iniciais preparam apenas um estado-base com peso de Hamming $k$;
- cada parâmetro recebe informações como posição, distância entre qubits e tipo de bloco.

O número esperado de parâmetros é

$$
N_\theta=\frac{k(2n-k-1)}{2}.
$$

**Saídas principais:** `ansatz`, `structure_df`, `initial_x_qubits` e `N_PARAMETERS`.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL — CÓPIA DA CABEÇA 20.1
# ============================================================


def CY_parameterized(identifier):
    """Cria o bloco lógico CY com uma rotação controlada CRY(theta)."""
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    """Cria o bloco lógico CCY com rotações RY(theta) e controles CCX."""
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    """Número esperado de parâmetros do ansatz: k(2n-k-1)/2."""
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    """Constrói o ansatz e registra a origem lógica de cada parâmetro."""
    # Preserva o estado aleatório global para que a construção do ansatz não
    # altere outras rotinas aleatórias do notebook.
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)

        # Prepara um estado-base com exatamente k excitações; não injeta o ótimo clássico.
        initial_x_qubits = []
        for excitation_index in range(k_value):
            qubit = n_value - excitation_index - 1
            qc.x(qubit)
            initial_x_qubits.append(qubit)

        # Cada bloco criado gera um registro que depois será ligado ao theta_index real.
        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    # A ordem dos parâmetros é extraída do circuito decomposto, a mesma forma usada
    # nas avaliações por Statevector.
    decomposed = qc.decompose()
    ordered_parameters = list(decomposed.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(ordered_parameters)
    }
    structure_rows = []
    for record in records:
        record = record.copy()
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })

    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")

    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return decomposed, structure_df, tuple(sorted(initial_x_qubits))


ANSATZ_SEED = RANDOM_SEED + 100 * N_ASSETS + TARGET_K
ansatz, structure_df, initial_x_qubits = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, ANSATZ_SEED
)
N_PARAMETERS = int(ansatz.num_parameters)

print("ANSATZ_SEED =", ANSATZ_SEED)
print("n_parameters =", N_PARAMETERS)
print("qubits com X inicial =", initial_x_qubits)
print("estado-base inicial em ordem Qiskit =", "".join(
    "1" if qubit in initial_x_qubits else "0"
    for qubit in range(N_ASSETS - 1, -1, -1)
))


### Célula 9 — Mapa físico, lógico e financeiro de cada $\theta_j$

**Em termos simples:** esta célula percorre o circuito decomposto e identifica em qual operação física cada parâmetro aparece, quais qubits ele toca e em que posições do circuito ele é usado.

A periodicidade **estrutural da unidade** é definida por:

$$
T_j=
\begin{cases}
4\pi, & \text{se o parâmetro aparece em uma porta CRY},\\
2\pi, & \text{se aparece somente em RY}.
\end{cases}
$$

Uma `RY` muda apenas por uma fase global após $2\pi$. Em uma `CRY`, essa troca de sinal ocorre somente no setor em que o controle vale 1 e pode se tornar uma fase relativa observável; por isso o período seguro da unidade controlada é $4\pi$.

Isso explica por que `theta_2` e `theta_3`, que pertencem a blocos `CY/CRY`, são varridos de zero a $4\pi$. Os demais parâmetros mostrados pertencem a blocos `CCY/RY` e são varridos de zero a $2\pi$.

Entretanto, o observável específico $P(\mathcal{X}_{\mathrm{opt}})$ pode repetir após $2\pi$ mesmo quando a unidade tem período $4\pi$. Por isso a versão 20.6 compara numericamente a primeira e a segunda metade das curvas `CRY` nos 100 vetores.

Os **ativos não mudam durante uma varredura**. Cada $\theta_j$ está ligado a um bloco lógico fixo do ansatz e, portanto, a qubits/ativos fixos. Os nomes são diferentes entre `theta_17`, `theta_25` etc. porque são parâmetros de blocos diferentes, não porque o código esteja trocando ações entre os vetores.

**Saídas principais:** `parameter_map_df`, `parameter_occurrence_df` e `structural_audit`.

In [ ]:
# ============================================================
# 9. MAPA FÍSICO, LÓGICO E FINANCEIRO DOS PARÂMETROS
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    """Liga cada theta a operações primitivas, qubits, período e posição no circuito."""
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(parameter_order)
    }
    occurrence_rows = []

    # Percorre cada instrução do circuito decomposto e registra todas as ocorrências
    # de parâmetros nas expressões angulares das portas.
    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(
            int(ansatz.find_bit(qubit).index)
            for qubit in instruction.qubits
        )
        for parameter_slot, expression in enumerate(operation.params):
            for parameter in getattr(expression, "parameters", set()):
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    # Uma linha por ocorrência física de parâmetro.
    occurrence_df = pd.DataFrame(occurrence_rows)
    physical_rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        primitive_qubits = tuple(sorted({
            int(qubit)
            for qubit_tuple in group["primitive_qubits"]
            for qubit in qubit_tuple
        }))
        # RY(theta+2pi) difere apenas por fase global, mas CRY pode exigir 4pi porque
        # o sinal relativo entre os setores do qubit de controle é observável.
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        physical_rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "primitive_parameter_qubits": primitive_qubits,
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
        })

    # Une a descrição lógica do bloco à descrição física observada após decomposição.
    parameter_map = structure_df.merge(
        pd.DataFrame(physical_rows),
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("theta_index").reset_index(drop=True)

    def logical_qubits(row):
        """Retorna todos os qubits pertencentes ao bloco lógico CY ou CCY."""
        i_value, l_value = int(row["i"]), int(row["l"])
        if row["ansatz_gate_type"] == "CY":
            return tuple(sorted((i_value, l_value)))
        return tuple(sorted((i_value, i_value + 1, l_value)))

    parameter_map["logical_block_qubits"] = parameter_map.apply(logical_qubits, axis=1)
    parameter_map["logical_assets"] = parameter_map["logical_block_qubits"].map(
        lambda qubits: tuple(tickers[index] for index in qubits)
    )
    parameter_map["block_size"] = parameter_map["logical_block_qubits"].map(len)

    # Auditoria estrutural: qualquer inconsistência interrompe o experimento.
    checks = {
        "all_parameters_mapped": bool(len(parameter_map) == N_PARAMETERS),
        "CY_is_CRY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "primitive_physical_type",
        ].eq("CRY").all()),
        "CCY_is_RY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "primitive_physical_type",
        ].eq("RY").all()),
        "CY_has_two_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "block_size",
        ].eq(2).all()),
        "CCY_has_three_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "block_size",
        ].eq(3).all()),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")

    return parameter_map, occurrence_df, checks


parameter_map_df, parameter_occurrence_df, structural_audit = build_physical_parameter_map(
    ansatz, structure_df
)

# Liga cada bloco aos pares clássicos contidos nos seus qubits lógicos.
def pair_metrics_for_block(qubits):
    """Resume os gaps clássicos dos pares de ativos internos ao bloco."""
    pairs = {tuple(sorted(pair)) for pair in combinations(qubits, 2)}
    subset = pair_gap_df.loc[
        pair_gap_df.apply(
            lambda row: tuple(sorted((int(row["asset_index_i"]), int(row["asset_index_j"])))) in pairs,
            axis=1,
        )
    ]
    return pd.Series({
        "n_internal_asset_pairs": int(len(subset)),
        "max_internal_G_ij": float(subset["G_ij"].max()),
        "mean_internal_G_ij": float(subset["G_ij"].mean()),
        "max_internal_joint_gap": float(subset["joint_flip_gap"].max()),
        "max_internal_nonadditivity_abs": float(subset["joint_flip_nonadditivity"].abs().max()),
        "internal_pairs": tuple(subset["pair"].tolist()),
    })

block_pair_metrics = parameter_map_df["logical_block_qubits"].apply(pair_metrics_for_block)
parameter_map_df = pd.concat([parameter_map_df, block_pair_metrics], axis=1)

print(json.dumps(structural_audit, indent=2, ensure_ascii=False))
display(parameter_map_df)


## Auditoria visual da criação do circuito

A tabela anterior é a ponte entre o índice local e a descrição transferível:

- `theta_index`: posição local neste circuito;
- `ansatz_gate_type`: bloco lógico `CY` ou `CCY`;
- `primitive_physical_type`: operação parametrizada observada após decomposição;
- `logical_block_qubits`: qubits realmente envolvidos pelo bloco;
- `logical_assets`: ativos associados a esses qubits;
- `max_internal_G_ij`: maior rigidez clássica entre os pares internos do bloco;
- `angular_period`: domínio máximo usado na varredura.

A célula seguinte mostra o circuito e contabiliza as operações. O estado preparado pelas portas `X` é apenas a semente de peso $k$, não o bitstring ótimo clássico.



### Célula 10 — Auditoria visual e decomposição somente para desenho

**Em termos simples:** esta célula mostra a estrutura simbólica do ansatz e cria uma cópia destinada exclusivamente à visualização.

A cópia visual decompõe as portas `CRY` em rotações `RY` e controles `CX`, mantendo as portas `CCX`. Isso produz um desenho semelhante ao circuito de referência, com:

- portas `X` responsáveis pelo estado inicial;
- rotações `RY` explícitas;
- controles `CX` e `CCX` visíveis;
- nenhuma mudança no objeto `ansatz` usado pelo `Statevector` e pelas varreduras.

Nesta etapa os ângulos ainda são simbólicos. Depois da seleção dos 100 vetores, outra célula atribui valores numéricos a um vetor real e desenha a versão numérica.


In [ ]:

# ============================================================
# 10. VISUALIZAÇÃO SIMBÓLICA — DECOMPOSIÇÃO SOMENTE PARA DESENHO
# ============================================================

# Contagem de portas do circuito que continua sendo usado nas avaliações.
operation_counts_original = pd.DataFrame(
    sorted(ansatz.count_ops().items()),
    columns=["operation", "count"],
)

# Cópia de apresentação: decompõe somente CRY. As portas CCX permanecem visíveis,
# evitando transformar o desenho em uma sequência extensa de H, T e CX.
try:
    ansatz_visual_symbolic = ansatz.decompose(gates_to_decompose=["cry"])
except TypeError:
    # Compatibilidade com versões do Qiskit que aceitam uma string isolada.
    ansatz_visual_symbolic = ansatz.decompose(gates_to_decompose="cry")

operation_counts_visual = pd.DataFrame(
    sorted(ansatz_visual_symbolic.count_ops().items()),
    columns=["operation", "count"],
)

print("PORTAS DO ANSATZ USADO NAS AVALIAÇÕES")
display(operation_counts_original)
print("PORTAS DA CÓPIA DE VISUALIZAÇÃO")
display(operation_counts_visual)

display(parameter_map_df[[
    "theta_index",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "distance",
    "first_instruction",
    "last_instruction",
    "angular_period",
    "internal_pairs",
    "max_internal_G_ij",
]])

# Desenho longo e horizontal, semelhante ao circuito mostrado como referência.
try:
    symbolic_figure = ansatz_visual_symbolic.draw(
        output="mpl",
        fold=-1,
        scale=0.72,
        idle_wires=False,
    )
    display(symbolic_figure)
    symbolic_figure_path = FIGURE_DIR / "ansatz_symbolic_cry_decomposed_visual.png"
    symbolic_figure.savefig(symbolic_figure_path, dpi=180, bbox_inches="tight")
    print("Figura simbólica salva em:", symbolic_figure_path.resolve())
except TypeError:
    # Fallback para versões com assinatura de draw mais restrita.
    symbolic_figure = ansatz_visual_symbolic.draw(output="mpl", fold=-1)
    display(symbolic_figure)
except Exception as exc:
    warnings.warn(
        f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}"
    )
    print(ansatz_visual_symbolic.draw(output="text", fold=160))


# Parte IV — compatibilidade entre `merge.pkl` e o circuito reconstruído

Antes de interpretar qualquer varredura, o notebook verifica:

1. todos os vetores possuem a dimensão esperada;
2. vetores salvos podem ser atribuídos ao circuito;
3. a distribuição exata recalculada é compatível com as colunas salvas;
4. nenhuma avaliação chama otimizador.


### Célula 11 — Avaliação exata de um vetor $\theta$

**Em termos simples:** esta é a função central usada em todas as intervenções posteriores. Ela recebe um vetor $\theta$, atribui os valores ao ansatz e calcula diretamente o estado quântico:

$$
|\psi(\theta)\rangle=U(\theta)|\psi_0\rangle.
$$

A função `evaluate_theta` devolve, entre outras métricas:

- probabilidade total dos bitstrings ótimos;
- energia esperada $\langle H\rangle$ e gap para a energia exata;
- bitstring dominante;
- massa dentro do subespaço válido de cardinalidade $k$;
- entropia e razão de participação;
- massa acumulada nos 1, 5 e 10 melhores portfólios clássicos.

Quando existe uma distribuição de referência, também são calculadas a distância de variação total

$$
\operatorname{TVD}(p,q)=\frac{1}{2}\sum_z|p_z-q_z|
$$

e a divergência de Jensen–Shannon.

Ao final, uma pequena amostra do banco é reavaliada para verificar compatibilidade entre os vetores salvos e o circuito reconstruído.


In [ ]:
# ============================================================
# 11. FUNÇÕES DE PARSE DOS VETORES E MÉTRICAS EXATAS
# ============================================================

# Converte todos os vetores salvos e mantém apenas aqueles compatíveis com o
# número de parâmetros do ansatz reconstruído.
best_parameters_column = RESOLVED_COLUMNS["best_parameters"]
merge_work_df = merge_df.copy()
merge_work_df["theta_vector"] = merge_work_df[best_parameters_column].map(parse_vector)
merge_work_df["theta_dimension"] = merge_work_df["theta_vector"].map(len)

dimension_counts = merge_work_df["theta_dimension"].value_counts().sort_index()
print("Dimensões encontradas:")
display(dimension_counts.rename_axis("theta_dimension").to_frame("rows"))

merge_work_df = merge_work_df.loc[
    merge_work_df["theta_dimension"].eq(N_PARAMETERS)
].copy()
if merge_work_df.empty:
    raise RuntimeError(
        f"Nenhum vetor do merge possui a dimensão esperada de {N_PARAMETERS} parâmetros."
    )

# Mapeamento entre índices do Statevector e bitstrings na convenção do Qiskit.
all_labels = np.asarray(
    [format(index, f"0{N_ASSETS}b") for index in range(2 ** N_ASSETS)],
    dtype=object,
)
label_to_index = {str(label): int(index) for index, label in enumerate(all_labels)}
valid_bitstrings = enumeration_df["bitstring_qiskit_order"].astype(str).to_numpy(dtype=object)
valid_indices = np.asarray([label_to_index[bitstring] for bitstring in valid_bitstrings], dtype=int)
valid_objectives = enumeration_df["objective"].to_numpy(dtype=float)
optimal_indices = np.asarray([label_to_index[bitstring] for bitstring in exact_qiskit_bitstrings], dtype=int)
optimal_set = set(exact_qiskit_bitstrings)


def total_variation_distance(p, q):
    """Calcula TVD(p,q)=0.5*sum(|p-q|), entre 0 e 1."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(0.5 * np.sum(np.abs(p - q)))


def jensen_shannon_divergence(p, q, epsilon=1e-15):
    """Calcula uma divergência simétrica e finita entre duas distribuições."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(p.sum(), epsilon)
    q = q / max(q.sum(), epsilon)
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + epsilon) / (m + epsilon)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + epsilon) / (m + epsilon)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


def circular_distance(a, b, period):
    """Menor distância entre dois ângulos em um círculo de período conhecido."""
    return float(abs((float(a) - float(b) + 0.5 * period) % period - 0.5 * period))


def shortest_delta_to_target(source, target, period):
    """Deslocamento assinado mais curto da origem até o alvo periódico."""
    return float((float(target) - float(source) + 0.5 * period) % period - 0.5 * period)


def statevector_from_theta(theta):
    """Retorna o vetor de estado complexo após atribuição direta dos parâmetros."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")
    assigned = ansatz.assign_parameters(theta, inplace=False)
    return np.asarray(Statevector.from_instruction(assigned).data, dtype=np.complex128)


def evaluate_theta(
    theta,
    reference_probability=None,
    return_valid_probability=False,
    return_statevector=False,
):
    """Atribui theta ao ansatz e calcula exatamente estado, energia e probabilidades."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")

    # Intervenção direta: não existe passo de otimização entre atribuir theta e avaliar.
    state_data = statevector_from_theta(theta)
    state = Statevector(state_data)
    probability = np.asarray(state.probabilities(), dtype=float)
    # Restringe a distribuição aos C(n,k) bitstrings que respeitam a cardinalidade.
    valid_probability = probability[valid_indices]
    valid_mass = float(valid_probability.sum())
    leakage = max(0.0, 1.0 - valid_mass)
    p_optimal = float(probability[optimal_indices].sum())
    # Energia física completa = valor esperado do operador + offset da conversão QUBO.
    energy = float(np.real(state.expectation_value(ising)) + ising_offset)
    dominant_index = int(np.argmax(probability))
    dominant_bitstring = str(all_labels[dominant_index])

    nonzero = valid_probability[valid_probability > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    normalized_entropy = float(entropy / np.log(len(valid_probability)))
    participation = float(1.0 / np.sum(valid_probability ** 2))

    dominant_valid_position = int(np.argmax(valid_probability))
    dominant_valid_rank = dominant_valid_position + 1

    result = {
        "p_optimal": p_optimal,
        "expected_energy": energy,
        "energy_gap": float(abs(energy - exact_energy)),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probability[dominant_index]),
        "dominant_is_optimal": bool(dominant_bitstring in optimal_set),
        "dominant_valid_rank": dominant_valid_rank,
        "p_top_1_classical": float(valid_probability[:1].sum()),
        "p_top_5_classical": float(valid_probability[:5].sum()),
        "p_top_10_classical": float(valid_probability[:10].sum()),
        "valid_dicke_mass": valid_mass,
        "leakage_outside_k": leakage,
        "normalized_entropy_valid": normalized_entropy,
        "participation_ratio_valid": participation,
    }
    if reference_probability is not None:
        result["tvd_vs_anchor"] = total_variation_distance(probability, reference_probability)
        result["jsd_vs_anchor"] = jensen_shannon_divergence(probability, reference_probability)
    if return_valid_probability:
        result["valid_probability"] = valid_probability.astype(np.float32)
    if return_statevector:
        result["statevector"] = state_data
    result["full_probability"] = probability
    return result


# Auditoria em uma amostra pequena antes das campanhas longas.
# Compara probabilidade e bitstring salvos com a reavaliação exata atual.
audit_indices = merge_work_df.sample(min(10, len(merge_work_df)), random_state=RANDOM_SEED).index
compatibility_rows = []
for row_index in audit_indices:
    row = merge_work_df.loc[row_index]
    metrics = evaluate_theta(row["theta_vector"])
    saved_p = (
        pd.to_numeric(pd.Series([row[RESOLVED_COLUMNS["p_best"]]]), errors="coerce").iloc[0]
        if RESOLVED_COLUMNS["p_best"] is not None
        else np.nan
    )
    saved_bit = (
        normalize_bitstring(row[RESOLVED_COLUMNS["dominant_bitstring"]])
        if RESOLVED_COLUMNS["dominant_bitstring"] is not None
        else None
    )
    compatibility_rows.append({
        "row_index": row_index,
        "saved_p": saved_p,
        "exact_p_recomputed": metrics["p_optimal"],
        "p_difference": metrics["p_optimal"] - saved_p if np.isfinite(saved_p) else np.nan,
        "saved_dominant": saved_bit,
        "exact_dominant_recomputed": metrics["dominant_bitstring"],
        "dominant_equal": bool(saved_bit == metrics["dominant_bitstring"]) if saved_bit else np.nan,
        "exact_energy_recomputed": metrics["expected_energy"],
        "leakage": metrics["leakage_outside_k"],
    })

compatibility_audit_df = pd.DataFrame(compatibility_rows)
display(compatibility_audit_df)

if compatibility_audit_df["leakage"].max() > 1e-10:
    raise RuntimeError("O circuito reconstruído apresentou vazamento para fora do subespaço k=4.")


### Célula 12 — Máscara dos 10% melhores e seleção de 100 vetores

**Em termos simples:** esta célula reduz o banco aos vetores mais promissores e escolhe 100 vetores completos para a campanha de varredura.

Para cada linha válida são construídos dois rankings percentuais:

- $R_p$: cresce quando a probabilidade salva da solução ótima aumenta;
- $R_{\mathrm{gap}}$: cresce quando o gap de energia diminui.

O score salvo é

$$
Q_{\mathrm{salvo}}=\frac{R_p+R_{\mathrm{gap}}}{2}.
$$

A máscara mantém aproximadamente os 10% superiores. Até 300 candidatos são reavaliados exatamente com `evaluate_theta`, e os 100 melhores pelo score exato são selecionados.

**Importante:** cada âncora é uma linha inteira do banco. A máscara não mistura componentes de vetores diferentes, não zera componentes internos e não muda os ativos associados a cada índice $\theta_j$.

In [ ]:
# ============================================================
# 12. MÁSCARA DOS 10% — 100 VETORES PARA AUDITORIA, 1 PARA VARREDURA
# ============================================================

objective_col = RESOLVED_COLUMNS["objective"]
best_objective_col = RESOLVED_COLUMNS["best_objective"]
p_col = RESOLVED_COLUMNS["p_best"]
gap_col = RESOLVED_COLUMNS["gap"]
status_col = RESOLVED_COLUMNS["status"]

bank = merge_work_df.copy()
bank["saved_p"] = (
    pd.to_numeric(bank[p_col], errors="coerce") if p_col is not None else np.nan
)
bank["saved_objective"] = (
    pd.to_numeric(bank[objective_col], errors="coerce") if objective_col is not None else np.nan
)

if gap_col is not None:
    bank["saved_gap"] = pd.to_numeric(bank[gap_col], errors="coerce").abs()
elif best_objective_col is not None and objective_col is not None:
    bank["saved_gap"] = (
        pd.to_numeric(bank[objective_col], errors="coerce")
        - pd.to_numeric(bank[best_objective_col], errors="coerce")
    ).abs()
else:
    bank["saved_gap"] = (bank["saved_objective"] - exact_energy).abs()

if status_col is not None:
    status_mask = bank[status_col].astype(str).str.lower().eq("ok")
else:
    status_mask = pd.Series(True, index=bank.index)

valid_bank = bank.loc[
    status_mask
    & bank["theta_vector"].notna()
    & bank["saved_gap"].notna()
].copy()

if valid_bank.empty:
    raise RuntimeError("Nenhum vetor válido foi encontrado para a seleção.")

if valid_bank["saved_p"].notna().any():
    valid_bank["p_quality_rank"] = valid_bank["saved_p"].rank(
        pct=True, ascending=True, method="average"
    )
else:
    valid_bank["p_quality_rank"] = 0.5

valid_bank["gap_quality_rank"] = valid_bank["saved_gap"].rank(
    pct=True, ascending=False, method="average"
)
valid_bank["quality_score_saved"] = 0.5 * (
    valid_bank["p_quality_rank"] + valid_bank["gap_quality_rank"]
)

threshold = float(valid_bank["quality_score_saved"].quantile(1.0 - TOP_FRACTION))
top_masked_bank = valid_bank.loc[
    valid_bank["quality_score_saved"].ge(threshold)
].copy()

if len(top_masked_bank) < N_AUDIT_VECTORS:
    raise RuntimeError(
        f"A máscara produziu {len(top_masked_bank)} vetores, mas a auditoria exige "
        f"{N_AUDIT_VECTORS}. Aumente TOP_FRACTION ou verifique o banco."
    )

candidate_count = min(
    max(int(MAX_EXACT_REEVALUATION), int(N_AUDIT_VECTORS)),
    len(top_masked_bank),
)
candidate_pool = top_masked_bank.sort_values(
    ["quality_score_saved", "saved_p", "saved_gap"],
    ascending=[False, False, True],
).head(candidate_count)

exact_candidate_rows = []
for row_index, row in candidate_pool.iterrows():
    row_hash = problem_fingerprint(row)
    if row_hash != DATA_HASH:
        raise RuntimeError(
            f"A linha {row_index} pertence a outro Hamiltoniano: {row_hash} != {DATA_HASH}."
        )
    metrics = evaluate_theta(row["theta_vector"])
    exact_candidate_rows.append({
        "source_row_index": row_index,
        "problem_hash": row_hash,
        "theta_vector": np.asarray(row["theta_vector"], dtype=float).copy(),
        "saved_p": row["saved_p"],
        "saved_gap": row["saved_gap"],
        "quality_score_saved": row["quality_score_saved"],
        **{key: value for key, value in metrics.items() if key != "full_probability"},
    })

exact_candidates_df = pd.DataFrame(exact_candidate_rows)
exact_candidates_df["p_rank_exact"] = exact_candidates_df["p_optimal"].rank(
    pct=True, ascending=True
)
exact_candidates_df["gap_rank_exact"] = exact_candidates_df["energy_gap"].rank(
    pct=True, ascending=False
)
exact_candidates_df["quality_score_exact"] = 0.5 * (
    exact_candidates_df["p_rank_exact"] + exact_candidates_df["gap_rank_exact"]
)

# Os 100 melhores são mantidos apenas para auditar a distribuição dos vetores.
audit_anchors_df = exact_candidates_df.sort_values(
    ["quality_score_exact", "p_optimal", "energy_gap"],
    ascending=[False, False, True],
).head(N_AUDIT_VECTORS).reset_index(drop=True)
audit_anchors_df.insert(
    0, "audit_vector_id", np.arange(len(audit_anchors_df), dtype=int)
)

if len(audit_anchors_df) != N_AUDIT_VECTORS:
    raise RuntimeError(
        f"Esperados {N_AUDIT_VECTORS} vetores de auditoria; "
        f"encontrados {len(audit_anchors_df)}."
    )
if audit_anchors_df["problem_hash"].nunique() != 1:
    raise RuntimeError("Os vetores de auditoria não pertencem ao mesmo Hamiltoniano.")

# Para a varredura, usa apenas o vetor de maior p_optimal; em empate, menor gap.
anchors_df = audit_anchors_df.sort_values(
    ["p_optimal", "energy_gap", "quality_score_exact", "audit_vector_id"],
    ascending=[False, True, False, True],
).head(1).copy().reset_index(drop=True)
anchors_df.insert(0, "anchor_id", np.arange(len(anchors_df), dtype=int))

if len(anchors_df) != 1:
    raise RuntimeError("A versão 20.11 exige exatamente um vetor para a varredura.")

# -----------------------------------------------------------------
# Auditoria dedicada da distribuição de theta_17 entre os 100 vetores
# -----------------------------------------------------------------
audit_theta_matrix = np.vstack(audit_anchors_df["theta_vector"].to_list())
if audit_theta_matrix.shape != (N_AUDIT_VECTORS, N_PARAMETERS):
    raise RuntimeError(
        f"Matriz theta de auditoria com forma {audit_theta_matrix.shape}; "
        f"esperado {(N_AUDIT_VECTORS, N_PARAMETERS)}."
    )

theta17_values = audit_theta_matrix[:, 17].astype(float)
theta17_audit_df = pd.DataFrame({
    "audit_vector_id": audit_anchors_df["audit_vector_id"].to_numpy(dtype=int),
    "source_row_index": audit_anchors_df["source_row_index"].to_numpy(),
    "p_optimal": audit_anchors_df["p_optimal"].to_numpy(dtype=float),
    "theta_17_raw": theta17_values,
    "theta_17_canonical_2pi": np.mod(theta17_values, 2 * np.pi),
})

theta17_summary_df = pd.DataFrame([{
    "n_vectors": int(len(theta17_values)),
    "theta_17_raw_min": float(np.min(theta17_values)),
    "theta_17_raw_max": float(np.max(theta17_values)),
    "theta_17_raw_range": float(np.ptp(theta17_values)),
    "theta_17_raw_mean": float(np.mean(theta17_values)),
    "theta_17_raw_std": float(np.std(theta17_values, ddof=1)),
    "n_unique_raw_3dp": int(pd.Series(theta17_values).round(3).nunique()),
    "n_unique_raw_6dp": int(pd.Series(theta17_values).round(6).nunique()),
    "n_unique_raw_9dp": int(pd.Series(theta17_values).round(9).nunique()),
    "all_equal_at_3dp": bool(pd.Series(theta17_values).round(3).nunique() == 1),
    "all_equal_at_6dp": bool(pd.Series(theta17_values).round(6).nunique() == 1),
}])

print("SELEÇÃO DO EXPERIMENTO")
print("Vetores mantidos para auditoria:", len(audit_anchors_df))
print("Vetores usados na varredura:", len(anchors_df))
display(anchors_df.drop(columns=["theta_vector"], errors="ignore"))

print("AUDITORIA DE theta_17 NOS 100 VETORES")
display(theta17_summary_df)
display(theta17_audit_df.sort_values("theta_17_raw").reset_index(drop=True))

audit_anchors_df.to_pickle(TABLE_DIR / "audit_100_selected_vectors.pkl")
theta17_audit_df.to_csv(TABLE_DIR / "theta17_values_100_vectors.csv", index=False)
theta17_summary_df.to_csv(TABLE_DIR / "theta17_summary_100_vectors.csv", index=False)



### Célula 12B — Circuito numérico no estilo da figura de referência

Esta célula escolhe, apenas para apresentação, o vetor com maior `p_optimal` entre os 100 vetores selecionados. Em seguida:

1. atribui os 30 valores de $\theta$ ao ansatz original;
2. decompõe somente as portas `CRY` em `RY` e `CX`;
3. mantém `CCX`, `CX`, `RY` e as portas `X` iniciais visíveis;
4. desenha o circuito horizontalmente, com os valores numéricos das rotações;
5. calcula a fidelidade entre o circuito original ligado e a cópia de visualização.

Essa célula **não substitui `ansatz`**, não altera `anchors_df` e não muda nenhuma varredura.


In [ ]:

# ============================================================
# 12B. CIRCUITO NUMÉRICO DECOMPOSTO — SOMENTE VISUALIZAÇÃO
# ============================================================

# Escolhe uma âncora representativa: maior probabilidade ótima exata e, em caso
# de empate, menor gap energético e menor anchor_id.
visual_anchor_row = anchors_df.sort_values(
    ["p_optimal", "energy_gap", "anchor_id"],
    ascending=[False, True, True],
).iloc[0]

visual_anchor_id = int(visual_anchor_row["anchor_id"])
visual_theta = np.asarray(visual_anchor_row["theta_vector"], dtype=float).copy()

if visual_theta.shape != (N_PARAMETERS,):
    raise RuntimeError(
        f"Vetor visual com dimensão {visual_theta.shape}; esperado ({N_PARAMETERS},)."
    )

# Circuito original numericamente ligado. Ele não é modificado in-place.
visual_bound_original = ansatz.assign_parameters(visual_theta, inplace=False)

# Cópia visual: decompõe apenas CRY -> RY/CX, preservando CCX.
try:
    visual_bound_decomposed = visual_bound_original.decompose(
        gates_to_decompose=["cry"]
    )
except TypeError:
    visual_bound_decomposed = visual_bound_original.decompose(
        gates_to_decompose="cry"
    )

# Auditoria física: as duas representações devem produzir o mesmo estado,
# salvo diferenças numéricas e uma possível fase global.
state_original_visual = np.asarray(
    Statevector.from_instruction(visual_bound_original).data,
    dtype=np.complex128,
)
state_decomposed_visual = np.asarray(
    Statevector.from_instruction(visual_bound_decomposed).data,
    dtype=np.complex128,
)
visual_state_fidelity = float(
    abs(np.vdot(state_original_visual, state_decomposed_visual)) ** 2
)

if not np.isclose(visual_state_fidelity, 1.0, atol=1e-10, rtol=1e-10):
    raise RuntimeError(
        "A decomposição destinada ao desenho alterou o estado: "
        f"fidelidade={visual_state_fidelity:.16f}."
    )

visual_anchor_summary_df = pd.DataFrame([{
    "anchor_id": visual_anchor_id,
    "source_row_index": visual_anchor_row["source_row_index"],
    "p_optimal": float(visual_anchor_row["p_optimal"]),
    "energy_gap": float(visual_anchor_row["energy_gap"]),
    "n_parameters": int(N_PARAMETERS),
    "state_fidelity_original_vs_visual": visual_state_fidelity,
    "visual_only": True,
}])

print("VETOR USADO SOMENTE PARA DESENHAR O CIRCUITO NUMÉRICO")
display(visual_anchor_summary_df)
print("Valores theta usados no desenho:")
display(pd.DataFrame({
    "theta_index": np.arange(N_PARAMETERS, dtype=int),
    "theta_value": visual_theta,
    "theta_over_pi": visual_theta / np.pi,
}))

print("Contagem de portas após decompor CRY para a figura:")
display(pd.DataFrame(
    sorted(visual_bound_decomposed.count_ops().items()),
    columns=["operation", "count"],
))

try:
    numeric_circuit_figure = visual_bound_decomposed.draw(
        output="mpl",
        fold=-1,
        scale=0.72,
        idle_wires=False,
    )
    display(numeric_circuit_figure)
    numeric_circuit_path = FIGURE_DIR / (
        f"ansatz_numeric_anchor_{visual_anchor_id:03d}_cry_decomposed.png"
    )
    numeric_circuit_figure.savefig(
        numeric_circuit_path,
        dpi=180,
        bbox_inches="tight",
    )
    print("Figura numérica salva em:", numeric_circuit_path.resolve())
except TypeError:
    numeric_circuit_figure = visual_bound_decomposed.draw(
        output="mpl",
        fold=-1,
    )
    display(numeric_circuit_figure)
except Exception as exc:
    warnings.warn(
        f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}"
    )
    print(visual_bound_decomposed.draw(output="text", fold=160))

visual_anchor_summary_df.to_csv(
    TABLE_DIR / "numeric_circuit_visualization_anchor_summary.csv",
    index=False,
)



# Parte V — varredura individual em um único vetor

Os 100 vetores selecionados continuam disponíveis em `audit_anchors_df`, mas não são todos varridos. A varredura utiliza somente o vetor representativo salvo em `anchors_df`.

Para cada parâmetro testado, apenas um componente é alterado. Os outros 29 parâmetros permanecem exatamente iguais aos valores do vetor escolhido.

Parâmetros detalhados:

$$
\{17,2,14,19,22,25,27,3,24\}.
$$

Não existe média, mediana ou envelope entre vetores nesta versão. Cada curva pertence ao mesmo vetor completo.



### Célula 13 — varredura simétrica de um parâmetro por vez no vetor escolhido

Para o único vetor $\boldsymbol{\theta}^{(0)}\in\mathbb{R}^{30}$ e para cada índice $j\in\mathcal{J}$, a célula avalia

$$
\boldsymbol{\theta}^{(0,j)}(\phi)
=
\bigl(
\theta_0^{(0)},\ldots,\theta_{j-1}^{(0)},
\phi,
\theta_{j+1}^{(0)},\ldots,\theta_{29}^{(0)}
\bigr).
$$

A janela é simétrica:

- `RY`: $-2\pi\leq\phi\leq2\pi$;
- `CRY`: $-4\pi\leq\phi\leq4\pi$.

O ponto original bruto é inserido exatamente na grade. Os checkpoints antigos não são reutilizados, porque a identidade do vetor de varredura é auditada nesta versão.


In [ ]:
# ============================================================
# 13. MOTOR DE VARREDURA INDIVIDUAL — UM VETOR, SEM COBYLA
# ============================================================


def inclusive_grid(start, end, step=None, n_points=None):
    """Cria uma grade que inclui explicitamente os limites start e end."""
    start, end = float(start), float(end)
    if step is not None:
        values = np.arange(start, end, float(step), dtype=float)
        if len(values) == 0 or not np.isclose(values[-1], end, atol=1e-12, rtol=0.0):
            values = np.append(values, end)
        else:
            values[-1] = end
        return values
    if n_points is None or n_points < 2:
        raise ValueError("Informe step ou n_points >= 2.")
    return np.linspace(start, end, int(n_points), endpoint=True)


def parameter_period(theta_index):
    """Recupera em parameter_map_df o período estrutural 2pi ou 4pi."""
    row = parameter_map_df.loc[
        parameter_map_df["theta_index"].eq(int(theta_index))
    ]
    if len(row) != 1:
        raise KeyError(f"theta_{theta_index} não foi identificado de forma única.")
    return float(row.iloc[0]["angular_period"])


def parameter_sweep_bounds(theta_index):
    """Usa uma janela simétrica [-T_j, T_j] para visualizar senos/cossenos."""
    period = parameter_period(theta_index)
    return float(-period), float(period)


def run_single_parameter_task(
    anchor_row,
    theta_index,
    n_points=SWEEP_POINTS_PER_PERIOD,
):
    """Varre um theta de uma âncora, mantendo todos os demais componentes fixos."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()

    period = parameter_period(theta_index)
    original_raw = float(theta_anchor[theta_index])
    original_canonical = float(original_raw % period)

    # Grade regular na janela simétrica [-T_j, T_j] + valor original inserido exatamente.
    sweep_start, sweep_end = parameter_sweep_bounds(theta_index)
    sweep_span = float(sweep_end - sweep_start)
    base_grid = inclusive_grid(sweep_start, sweep_end, n_points=int(n_points))
    grid = np.unique(np.append(base_grid, original_raw))
    grid.sort()
    original_grid_index = int(np.argmin(np.abs(grid - original_raw)))
    if not np.isclose(
        grid[original_grid_index], original_raw, atol=1e-14, rtol=0.0
    ):
        raise RuntimeError(f"Não foi possível inserir o ponto original bruto de theta_{theta_index}.")

    task_stem = f"anchor_{anchor_id:03d}_theta_{theta_index:02d}"
    summary_path = CHECKPOINT_DIR / f"detailed_{task_stem}.pkl"
    distribution_path = CHECKPOINT_DIR / f"detailed_{task_stem}_valid_probabilities.npz"

    # Procura primeiro na versão 20.7 e depois na 20.6. O conteúdo das
    # varreduras é compatível; apenas os resumos e gráficos foram corrigidos.
    if REUSE_SWEEP_CHECKPOINTS:
        candidate_directories = [CHECKPOINT_DIR]
        if LEGACY_CHECKPOINT_DIR != CHECKPOINT_DIR:
            candidate_directories.append(LEGACY_CHECKPOINT_DIR)

        for candidate_directory in candidate_directories:
            candidate_summary = candidate_directory / f"detailed_{task_stem}.pkl"
            candidate_distribution = (
                candidate_directory / f"detailed_{task_stem}_valid_probabilities.npz"
            )
            if not (candidate_summary.exists() and candidate_distribution.exists()):
                continue

            cached = pd.read_pickle(candidate_summary)
            required = {
                "anchor_id", "theta_index", "is_original_theta", "parameter_name",
                "theta_value", "p_optimal", "period",
            }
            if required.issubset(cached.columns):
                return cached, candidate_distribution

    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]
    parameter_name = str(list(ansatz.parameters)[theta_index])

    rows = []
    probability_matrix = np.empty((len(grid), len(valid_indices)), dtype=np.float32)

    for grid_index, value in enumerate(grid):
        theta_test = theta_anchor.copy()
        theta_test[theta_index] = float(value)

        changed_indices = np.flatnonzero(
            ~np.isclose(theta_test, theta_anchor, atol=1e-14, rtol=0.0)
        )
        is_original = bool(grid_index == original_grid_index)
        if is_original:
            if len(changed_indices) != 0 and not (
                len(changed_indices) == 1 and int(changed_indices[0]) == theta_index
            ):
                raise RuntimeError(
                    f"Auditoria falhou no ponto original de theta_{theta_index}, "
                    f"âncora {anchor_id}: índices alterados = {changed_indices.tolist()}"
                )
        elif changed_indices.tolist() != [theta_index]:
            raise RuntimeError(
                f"Auditoria falhou em theta_{theta_index}, âncora {anchor_id}: "
                f"índices alterados = {changed_indices.tolist()}"
            )

        metrics = evaluate_theta(
            theta_test,
            reference_probability=reference_probability,
            return_valid_probability=True,
        )
        probability_matrix[grid_index] = metrics.pop("valid_probability")
        metrics.pop("full_probability")

        rows.append({
            "anchor_id": anchor_id,
            "source_row_index": anchor_row["source_row_index"],
            "theta_index": theta_index,
            "parameter_name": parameter_name,
            "grid_index": int(grid_index),
            "theta_value": float(value),
            "theta_over_period": float(value / period),
            "theta_over_pi": float(value / np.pi),
            "theta_over_sweep_window": float((value - sweep_start) / sweep_span),
            "sweep_start": float(sweep_start),
            "sweep_end": float(sweep_end),
            "sweep_span": float(sweep_span),
            "period": period,
            "anchor_theta_raw": original_raw,
            "anchor_theta_canonical": original_canonical,
            "is_original_theta": is_original,
            "distance_to_anchor": circular_distance(value, original_raw, period),
            **metrics,
            "optimizer_used": False,
        })

    task_df = pd.DataFrame(rows)

    original_rows = task_df.loc[task_df["is_original_theta"]]
    if len(original_rows) != 1:
        raise RuntimeError(
            f"Esperado um ponto original para theta_{theta_index}, âncora {anchor_id}; "
            f"encontrados {len(original_rows)}."
        )
    original_p = float(original_rows.iloc[0]["p_optimal"])
    if not np.isclose(original_p, anchor_metrics["p_optimal"], atol=1e-12, rtol=1e-10):
        raise RuntimeError(
            f"O ponto original de theta_{theta_index}, âncora {anchor_id}, não reproduziu "
            f"P(X_opt): {original_p} versus {anchor_metrics['p_optimal']}"
        )

    p_min = float(task_df["p_optimal"].min())
    p_max = float(task_df["p_optimal"].max())
    span = p_max - p_min
    flat_threshold = max(
        SWEEP_FLAT_ABS_TOL,
        SWEEP_FLAT_REL_TOL * max(abs(p_max), 1.0),
    )
    task_df["p_span"] = span
    task_df["flat_threshold"] = flat_threshold
    task_df["is_flat_sweep"] = bool(span <= flat_threshold)

    task_df.to_pickle(summary_path)
    np.savez_compressed(
        distribution_path,
        valid_probability=probability_matrix,
        theta_grid=grid,
        valid_bitstrings=valid_bitstrings,
        valid_objectives=valid_objectives,
    )
    return task_df, distribution_path


if len(anchors_df) != N_ANCHORS:
    raise RuntimeError(
        f"A campanha exige {N_ANCHORS} vetores selecionados; encontrados {len(anchors_df)}."
    )

for theta_index in DETAILED_THETA_INDICES:
    if not 0 <= theta_index < N_PARAMETERS:
        raise IndexError(f"theta_{theta_index} não existe no circuito com {N_PARAMETERS} parâmetros.")

all_detailed_frames = []
detailed_distribution_paths = {}
total_tasks = len(anchors_df) * len(DETAILED_THETA_INDICES)
completed_tasks = 0

for _, anchor_row in anchors_df.iterrows():
    anchor_id = int(anchor_row["anchor_id"])
    for theta_index in DETAILED_THETA_INDICES:
        task_df, distribution_path = run_single_parameter_task(anchor_row, theta_index)
        all_detailed_frames.append(task_df)
        detailed_distribution_paths[(anchor_id, int(theta_index))] = distribution_path
        completed_tasks += 1
        if completed_tasks == 1 or completed_tasks % 25 == 0 or completed_tasks == total_tasks:
            print(
                f"tarefas concluídas: {completed_tasks}/{total_tasks} | "
                f"anchor={anchor_id:03d} | theta_{theta_index}"
            )

individual_sweep_df = pd.concat(all_detailed_frames, ignore_index=True)
individual_sweep_path = TABLE_DIR / "individual_sweeps_single_vector.pkl"
individual_sweep_df.to_pickle(individual_sweep_path)

print("Âncoras avaliadas:", individual_sweep_df["anchor_id"].nunique())
print("Thetas avaliados:", sorted(individual_sweep_df["theta_index"].unique().tolist()))
print("Total de avaliações detalhadas:", len(individual_sweep_df))
print("Tabela salva em:", individual_sweep_path.resolve())


### Célula 14 — curvas brutas do único vetor e auditoria de `theta_17`

Esta célula não reúne 100 curvas. Para cada parâmetro, ela mostra apenas a curva do vetor escolhido:

- eixo horizontal bruto e simétrico;
- ponto original bruto;
- mínimo e máximo da curva;
- linha de referência em $P=0.90$;
- nenhuma normalização min–max;
- nenhuma mediana ou envelope entre vetores.

A tabela de `theta_17` construída na Célula 12 continua mostrando os valores provenientes dos 100 vetores, permitindo verificar se a aparente igualdade é exata ou apenas aproximada.


In [ ]:
# ============================================================
# 14. CURVAS BRUTAS DO ÚNICO VETOR E EXTREMOS
# ============================================================

if individual_sweep_df["anchor_id"].nunique() != 1:
    raise RuntimeError(
        "A versão 20.11 espera exatamente um vetor no resultado da varredura."
    )

parameter_order = list(ansatz.parameters)
if len(parameter_order) != N_PARAMETERS:
    raise RuntimeError("A ordem dos parâmetros do ansatz está inconsistente.")

selected_parameter_map_df = parameter_map_df.loc[
    parameter_map_df["theta_index"].isin(DETAILED_THETA_INDICES)
].copy().sort_values("theta_index")
selected_parameter_map_df["parameter_name_from_ansatz"] = selected_parameter_map_df[
    "theta_index"
].map(lambda index: str(parameter_order[int(index)]))

mapping_audit_df = selected_parameter_map_df[[
    "theta_index",
    "parameter_name_from_ansatz",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "angular_period",
]].copy()
mapping_audit_df["period_over_pi"] = mapping_audit_df["angular_period"] / np.pi
mapping_audit_df["mapping_depends_on_anchor"] = False

display(mapping_audit_df)
mapping_audit_df.to_csv(TABLE_DIR / "fixed_theta_asset_mapping.csv", index=False)

original_points_df = individual_sweep_df.loc[
    individual_sweep_df["is_original_theta"]
].copy()
if len(original_points_df) != len(DETAILED_THETA_INDICES):
    raise RuntimeError(
        f"Esperados {len(DETAILED_THETA_INDICES)} pontos originais; "
        f"encontrados {len(original_points_df)}."
    )

anchor_probability_unique_df = pd.DataFrame([{
    "anchor_id": int(original_points_df["anchor_id"].iloc[0]),
    "p_original": float(original_points_df["p_optimal"].mean()),
}])

# Extremos de cada uma das nove curvas.
sweep_extrema_rows = []
for theta_index, group in individual_sweep_df.groupby("theta_index", sort=True):
    group = group.sort_values("theta_value").copy()
    original_row = group.loc[group["is_original_theta"]]
    if len(original_row) != 1:
        raise RuntimeError(f"Ponto original inconsistente para theta_{theta_index}.")
    original_row = original_row.iloc[0]
    min_row = group.loc[group["p_optimal"].idxmin()]
    max_row = group.loc[group["p_optimal"].idxmax()]
    sweep_extrema_rows.append({
        "anchor_id": int(group["anchor_id"].iloc[0]),
        "theta_index": int(theta_index),
        "period": float(group["period"].iloc[0]),
        "sweep_start": float(group["sweep_start"].iloc[0]),
        "sweep_end": float(group["sweep_end"].iloc[0]),
        "theta_original_raw": float(original_row["anchor_theta_raw"]),
        "p_original": float(original_row["p_optimal"]),
        "theta_at_p_min": float(min_row["theta_value"]),
        "p_sweep_min": float(min_row["p_optimal"]),
        "theta_at_p_max": float(max_row["theta_value"]),
        "p_sweep_max": float(max_row["p_optimal"]),
        "p_sweep_range": float(max_row["p_optimal"] - min_row["p_optimal"]),
        "is_flat_sweep": bool(group["is_flat_sweep"].iloc[0]),
    })

sweep_extrema_per_vector_df = pd.DataFrame(sweep_extrema_rows)
display(sweep_extrema_per_vector_df)
sweep_extrema_per_vector_df.to_csv(
    TABLE_DIR / "single_vector_sweep_extrema.csv", index=False
)

# Figura principal: uma curva por theta, usando diretamente theta_value bruto.
n_theta = len(DETAILED_THETA_INDICES)
n_cols = 3
n_rows = int(np.ceil(n_theta / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4.8 * n_rows), squeeze=False)
axes_flat = axes.ravel()

for panel_index, theta_index in enumerate(DETAILED_THETA_INDICES):
    ax = axes_flat[panel_index]
    group = individual_sweep_df.loc[
        individual_sweep_df["theta_index"].eq(theta_index)
    ].sort_values("theta_value").copy()
    map_row = selected_parameter_map_df.loc[
        selected_parameter_map_df["theta_index"].eq(theta_index)
    ].iloc[0]

    period = float(group["period"].iloc[0])
    sweep_start = float(group["sweep_start"].iloc[0])
    sweep_end = float(group["sweep_end"].iloc[0])
    x = group["theta_value"].to_numpy(dtype=float)
    y = group["p_optimal"].to_numpy(dtype=float)
    original_row = group.loc[group["is_original_theta"]].iloc[0]
    extrema_row = sweep_extrema_per_vector_df.loc[
        sweep_extrema_per_vector_df["theta_index"].eq(theta_index)
    ].iloc[0]

    ax.plot(x, y, linewidth=1.8, label="varredura do vetor único")
    ax.scatter(
        [float(original_row["anchor_theta_raw"])],
        [float(original_row["p_optimal"])],
        s=38,
        zorder=5,
        label="valor original bruto",
    )
    ax.scatter(
        [float(extrema_row["theta_at_p_max"])],
        [float(extrema_row["p_sweep_max"])],
        marker="^",
        s=38,
        zorder=5,
        label="máximo",
    )
    ax.scatter(
        [float(extrema_row["theta_at_p_min"])],
        [float(extrema_row["p_sweep_min"])],
        marker="v",
        s=38,
        zorder=5,
        label="mínimo",
    )
    ax.axhline(0.90, linestyle="--", linewidth=1.0, alpha=0.7)
    ax.axvline(0.0, linestyle=":", linewidth=1.0, alpha=0.7)

    if np.isclose(period, 4 * np.pi):
        ticks = [-4*np.pi, -2*np.pi, 0.0, 2*np.pi, 4*np.pi]
        tick_labels = [r"$-4\pi$", r"$-2\pi$", "0", r"$2\pi$", r"$4\pi$"]
    else:
        ticks = [-2*np.pi, -np.pi, 0.0, np.pi, 2*np.pi]
        tick_labels = [r"$-2\pi$", r"$-\pi$", "0", r"$\pi$", r"$2\pi$"]

    ax.set_xticks(ticks, tick_labels)
    ax.set_xlim(sweep_start, sweep_end)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel(f"valor bruto de theta_{theta_index} (rad)")
    ax.set_ylabel(r"$P(\mathcal{X}_{\mathrm{opt}})$")
    ax.set_title(
        f"theta_{theta_index} | {map_row['ansatz_gate_type']}/"
        f"{map_row['primitive_physical_type']} | janela="
        f"[{sweep_start/np.pi:.0f}π,{sweep_end/np.pi:.0f}π]\n"
        f"qubits={map_row['logical_block_qubits']} | "
        f"ativos={map_row['logical_assets']}"
    )
    ax.grid(alpha=0.25)
    if panel_index == 0:
        ax.legend(loc="best", fontsize=8)

for unused_index in range(n_theta, len(axes_flat)):
    axes_flat[unused_index].axis("off")

focus_anchor = anchors_df.iloc[0]
fig.suptitle(
    "Varredura individual em um único vetor — eixo bruto e simétrico\n"
    f"anchor_id={int(focus_anchor['anchor_id'])} | "
    f"source_row_index={focus_anchor['source_row_index']} | "
    f"p_original={float(focus_anchor['p_optimal']):.6f}",
    fontsize=15,
    y=1.005,
)
fig.tight_layout()
single_vector_figure_path = FIGURE_DIR / "single_vector_symmetric_raw_sweeps.png"
fig.savefig(single_vector_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Figura salva em:", single_vector_figure_path.resolve())

# Auditoria de domínio: nenhuma curva pode começar em zero nesta versão.
domain_audit_df = individual_sweep_df.groupby("theta_index").agg(
    observed_theta_min=("theta_value", "min"),
    observed_theta_max=("theta_value", "max"),
    configured_sweep_start=("sweep_start", "first"),
    configured_sweep_end=("sweep_end", "first"),
).reset_index()
domain_audit_df["min_matches_configuration"] = np.isclose(
    domain_audit_df["observed_theta_min"],
    domain_audit_df["configured_sweep_start"],
    atol=1e-12,
    rtol=0.0,
)
domain_audit_df["max_matches_configuration"] = np.isclose(
    domain_audit_df["observed_theta_max"],
    domain_audit_df["configured_sweep_end"],
    atol=1e-12,
    rtol=0.0,
)
if not domain_audit_df[[
    "min_matches_configuration", "max_matches_configuration"
]].all().all():
    raise RuntimeError("O domínio observado não corresponde à janela simétrica configurada.")

display(domain_audit_df)
domain_audit_df.to_csv(TABLE_DIR / "single_vector_sweep_domain_audit.csv", index=False)


# Parte VI — teste coletivo e cumulativo dos 23 $\theta$ não ativos

## Objetivo do experimento

O circuito possui 30 parâmetros. Nesta etapa:

- os **7 parâmetros ativos** já definidos no notebook (`ACTIVE_THETA_INDICES`) são colocados nos seus **máximos individuais** encontrados na varredura (`theta_at_p_max`);
- os outros **23 parâmetros** são tratados como o conjunto a testar;
- nenhuma otimização é executada nesta parte.

> **Importante:** “melhor ponto” aqui significa o máximo obtido na varredura **individual** de cada um dos 7 parâmetros. Colocar os 7 máximos juntos não é uma nova otimização conjunta; por isso esse vetor é primeiro medido e passa a ser a referência do teste.

### Experimento A — intervenção coletiva

Partindo da referência com os 7 parâmetros fixos, os outros 23 são todos forçados para

\[
\theta_j=\pi/2.
\]

Depois do `assign_parameters`, medimos novamente:

- \(P(\mathcal{X}_{opt})\);
- energia esperada;
- bitstring dominante e sua probabilidade;
- distância de variação total (TVD) da distribuição;
- fidelidade com o statevector de referência.

### Experimento B — intervenção cumulativa aleatória

A ordem dos 23 índices é embaralhada com uma semente fixa. O valor de intervenção continua sendo \(\pi/2\), para não misturar o efeito de **qual parâmetro foi alterado** com o efeito de **qual valor foi escolhido**.

Começamos outra vez no vetor de referência:

1. força 1 dos 23 parâmetros para \(\pi/2\) e mede;
2. mantém essa alteração, força um segundo parâmetro e mede;
3. continua cumulativamente até os 23.

A última linha obrigatoriamente deve reproduzir o Experimento A.


In [ ]:
# ============================================================
# 17. REFERÊNCIA: 7 THETAS NOS MÁXIMOS INDIVIDUAIS
# ============================================================

INTERVENTION_VALUE = np.pi / 2
CHANGE_TOL = 1e-10

# Os 23 testados são exatamente o complemento dos 7 índices ativos.
INSENSITIVE_THETA_INDICES = sorted(
    set(range(N_PARAMETERS)) - set(ACTIVE_THETA_INDICES)
)
if len(ACTIVE_THETA_INDICES) != 7 or len(INSENSITIVE_THETA_INDICES) != 23:
    raise RuntimeError("A divisão esperada é 7 parâmetros ativos + 23 parâmetros testados.")

# Começa no MESMO vetor usado como âncora pela varredura.
theta_reference = np.asarray(anchors_df.iloc[0]["theta_vector"], dtype=float).copy()

# Substitui somente os 7 ativos pelos máximos das respectivas varreduras 1D.
best7 = sweep_extrema_per_vector_df.loc[
    sweep_extrema_per_vector_df["theta_index"].isin(ACTIVE_THETA_INDICES),
    ["theta_index", "theta_at_p_max", "p_sweep_max"],
].copy().sort_values("theta_index")

if set(best7["theta_index"].astype(int)) != set(ACTIVE_THETA_INDICES):
    raise RuntimeError("Nem todos os 7 theta ativos possuem theta_at_p_max disponível.")

for row in best7.itertuples(index=False):
    theta_reference[int(row.theta_index)] = float(row.theta_at_p_max)

# Auditoria: esta tabela mostra exatamente quais 7 valores ficaram fixos.
best7 = best7.rename(columns={"theta_at_p_max": "theta_fixed_in_reference"})
display(best7.reset_index(drop=True))

# Se algum dos 23 já estiver em pi/2 na referência, ele será selecionado pela ordem
# aleatória, mas esse passo específico não produzirá deslocamento numérico.
already_at_target = [
    i for i in INSENSITIVE_THETA_INDICES
    if np.isclose(theta_reference[i], INTERVENTION_VALUE, atol=1e-14, rtol=0.0)
]
print("23 theta testados:", INSENSITIVE_THETA_INDICES)
print("Já estavam em pi/2 na referência:", already_at_target)


In [ ]:
# ============================================================
# 18. MEDIÇÃO DIRETA — ASSIGN_PARAMETERS VISÍVEL
# ============================================================

def measure_theta(theta):
    """Atribui o vetor ao ansatz e mede somente o necessário para este experimento."""
    theta = np.asarray(theta, dtype=float)

    # LINHA-CHAVE: os 30 valores são realmente atribuídos ao circuito, sem otimizador.
    assigned = ansatz.assign_parameters(theta, inplace=False)

    # O Statevector é calculado diretamente do circuito já parametrizado.
    state = Statevector.from_instruction(assigned)
    probabilities = np.asarray(state.probabilities(), dtype=float)

    dominant_index = int(np.argmax(probabilities))
    return {
        "p_optimal": float(probabilities[optimal_indices].sum()),
        "expected_energy": float(np.real(state.expectation_value(ising)) + ising_offset),
        "dominant_bitstring": str(all_labels[dominant_index]),
        "dominant_probability": float(probabilities[dominant_index]),
        "probabilities": probabilities,
        "statevector": np.asarray(state.data, dtype=np.complex128),
    }


reference_metrics = measure_theta(theta_reference)
reference_probability = reference_metrics["probabilities"]
reference_statevector = reference_metrics["statevector"]


def comparison_row(label, theta, n_modified, last_theta=None, modified_indices=()):
    """Compara uma intervenção com a referência dos 7 máximos individuais."""
    metrics = measure_theta(theta)
    probability_delta = metrics["probabilities"] - reference_probability

    # TVD detecta qualquer redistribuição de probabilidade, mesmo sem trocar o bitstring dominante.
    tvd = float(0.5 * np.abs(probability_delta).sum())

    # Fidelidade detecta também mudanças do estado que podem ficar escondidas nas probabilidades.
    fidelity = float(abs(np.vdot(reference_statevector, metrics["statevector"])) ** 2)

    delta_p = float(metrics["p_optimal"] - reference_metrics["p_optimal"])
    delta_e = float(metrics["expected_energy"] - reference_metrics["expected_energy"])
    bit_changed = bool(metrics["dominant_bitstring"] != reference_metrics["dominant_bitstring"])
    actually_changed = np.flatnonzero(
        ~np.isclose(theta, theta_reference, atol=1e-14, rtol=0.0)
    ).astype(int).tolist()

    # Esta coluna diz ONDE a diferença apareceu, sem depender só de P(X_opt).
    signals = []
    if abs(delta_p) > CHANGE_TOL:
        signals.append("P_optimal")
    if abs(delta_e) > CHANGE_TOL:
        signals.append("energia")
    if bit_changed:
        signals.append("bitstring")
    if tvd > CHANGE_TOL:
        signals.append("distribuicao")
    if (1.0 - fidelity) > CHANGE_TOL:
        signals.append("estado")
    signature = "+".join(signals) if signals else "nenhuma"

    return {
        "label": label,
        "n_modified": int(n_modified),
        "last_theta_added": last_theta,
        "modified_indices": tuple(map(int, modified_indices)),
        "n_actually_changed": int(len(actually_changed)),
        "actually_changed_indices": tuple(actually_changed),
        "p_optimal": metrics["p_optimal"],
        "delta_p_optimal": delta_p,
        "expected_energy": metrics["expected_energy"],
        "delta_energy": delta_e,
        "dominant_bitstring": metrics["dominant_bitstring"],
        "dominant_probability": metrics["dominant_probability"],
        "bitstring_changed": bit_changed,
        "tvd_vs_reference": tvd,
        "state_fidelity_vs_reference": fidelity,
        "change_signature": signature,
        "any_detectable_change": bool(
            abs(delta_p) > CHANGE_TOL
            or abs(delta_e) > CHANGE_TOL
            or bit_changed
            or tvd > CHANGE_TOL
            or (1.0 - fidelity) > CHANGE_TOL
        ),
    }


In [ ]:
# ============================================================
# 19. EXPERIMENTO A — TODOS OS 23 THETAS = pi/2
# ============================================================

theta_all23 = theta_reference.copy()

# INTERVENÇÃO: somente os 23 índices testados recebem exatamente o mesmo valor pi/2.
theta_all23[INSENSITIVE_THETA_INDICES] = INTERVENTION_VALUE

# Garante que nenhum dos 7 parâmetros de controle foi alterado por acidente.
if not np.array_equal(
    theta_all23[ACTIVE_THETA_INDICES],
    theta_reference[ACTIVE_THETA_INDICES],
):
    raise RuntimeError("Um dos 7 theta ativos foi alterado no Experimento A.")

experiment_all23_df = pd.DataFrame([
    comparison_row("referencia_7_melhores", theta_reference, 0),
    comparison_row(
        "todos_23_em_pi_sobre_2",
        theta_all23,
        23,
        modified_indices=INSENSITIVE_THETA_INDICES,
    ),
])

display(experiment_all23_df)
experiment_all23_df.to_csv(
    TABLE_DIR / "experiment_all_23_theta_pi_over_2.csv",
    index=False,
)


In [ ]:
# ============================================================
# 20. EXPERIMENTO B — 1, 2, 3, ..., 23 ALTERAÇÕES CUMULATIVAS
# ============================================================

# A semente fixa torna a ordem aleatória totalmente reproduzível.
rng = np.random.default_rng(RANDOM_SEED)
random_order = rng.permutation(INSENSITIVE_THETA_INDICES).astype(int).tolist()

# Reinicia da referência; este experimento não herda o vetor do Experimento A.
theta_progressive = theta_reference.copy()
progressive_rows = [
    comparison_row("referencia_7_melhores", theta_progressive, 0)
]

for step, theta_index in enumerate(random_order, start=1):
    # A cada passo somente UM novo theta é acrescentado ao conjunto já modificado.
    theta_progressive[theta_index] = INTERVENTION_VALUE

    progressive_rows.append(
        comparison_row(
            f"passo_{step:02d}",
            theta_progressive,
            n_modified=step,
            last_theta=int(theta_index),
            modified_indices=random_order[:step],
        )
    )

progressive_23_df = pd.DataFrame(progressive_rows)

# A etapa 23 precisa ser exatamente o mesmo vetor do Experimento A.
if not np.array_equal(theta_progressive, theta_all23):
    raise RuntimeError("O passo 23 não reproduziu o vetor do Experimento A.")

display(progressive_23_df[[
    "n_modified",
    "n_actually_changed",
    "last_theta_added",
    "p_optimal",
    "delta_p_optimal",
    "expected_energy",
    "delta_energy",
    "dominant_bitstring",
    "bitstring_changed",
    "tvd_vs_reference",
    "state_fidelity_vs_reference",
    "change_signature",
    "any_detectable_change",
]])

progressive_23_df.to_csv(
    TABLE_DIR / "experiment_progressive_1_to_23_theta_pi_over_2.csv",
    index=False,
)

# Mostra o primeiro passo em que qualquer diferença mensurável aparece.
changed = progressive_23_df.loc[
    progressive_23_df["n_modified"].gt(0)
    & progressive_23_df["any_detectable_change"]
]
if changed.empty:
    print("Nenhuma mudança detectável apareceu em nenhum dos 23 passos.")
else:
    print("Primeiro passo com mudança detectável:")
    display(changed.head(1))

print("Ordem aleatória usada:", random_order)


# Parte VII — causalidade estrutural dos 7 \(\theta\) sensíveis

## Hipótese

A observação da Parte VI sugere que, **na arquitetura original**, 23 parâmetros são redundantes para o estado encontrado e somente os índices

\[
\theta_{2},\theta_{14},\theta_{17},\theta_{19},\theta_{22},\theta_{25},\theta_{27}
\]

controlam a solução observada.

Agora testamos se essa importância é explicada por:

- **identidade do bloco**: tipo `CY/CCY` e qubits tocados;
- **posição** do bloco no circuito;
- **ordem relativa** entre os 7 blocos;
- ou uma combinação desses fatores.

### Regra para evitar circularidade

O bitstring ótimo já é conhecido pela enumeração clássica, mas **não entra na função objetivo do otimizador**. Em todos os testes, o otimizador vê somente

\[
E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle.
\]

Somente depois da otimização medimos \(P(\mathcal X_{opt})\), bitstring dominante e distância de Hamming até o conjunto ótimo.


In [ ]:
# ============================================================
# 21. CONFIGURAÇÃO DO EXPERIMENTO 20.16 — TESTES CAUSAIS
# ============================================================

from scipy.optimize import minimize

STRUCTURAL_ROOT = OUTPUT_ROOT / "structural_causality_7theta"
STRUCTURAL_TABLE_DIR = STRUCTURAL_ROOT / "tables"
STRUCTURAL_FIGURE_DIR = STRUCTURAL_ROOT / "figures"
for directory in [STRUCTURAL_ROOT, STRUCTURAL_TABLE_DIR, STRUCTURAL_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reotimização principal: todos os 30 parâmetros ficam livres.
ARCHITECTURE_MAXITER = 700
ARCHITECTURE_RESTARTS = 3

# Varredura de posição: 7 blocos x 30 posições. Um restart mantém o custo controlado.
RUN_FULL_POSITION_SCAN = True
POSITION_SCAN_MAXITER = 300
POSITION_SCAN_RESTARTS = 1

# Um parâmetro é marcado como sensível se uma perturbação estrutural de T/4
# produzir mudança observável acima destes limites.
SENSITIVITY_PROB_TOL = 1e-6
SENSITIVITY_ENERGY_TOL = 1e-8
SENSITIVITY_TVD_TOL = 1e-6

print("Saídas causais 20.16:", STRUCTURAL_ROOT.resolve())
print("Reotimização das arquiteturas: 30 parâmetros livres.")
print("Varredura completa de posição:", RUN_FULL_POSITION_SCAN)


### Célula 22 — mapa dos 7 blocos e movimento necessário no bitstring

Esta célula responde primeiro à pergunta estrutural, sem otimização.

Para cada um dos 7 parâmetros ela mostra:

- posição original do bloco;
- tipo `CY` ou `CCY`;
- qubits e ativos tocados;
- primeira e última instrução física associada ao parâmetro.

Também compara o estado-base preparado pelas portas `X` com um bitstring ótimo de referência. Se existir mais de um ótimo degenerado, escolhe-se apenas para **visualização** aquele que já recebe maior probabilidade no estado de referência; o conjunto completo de ótimos continua sendo usado em `P_optimal`.


In [ ]:
# ============================================================
# 22. MAPA ESTRUTURAL DOS 7 BLOCOS + TRANSIÇÃO DE BITSTRING
# ============================================================

# Ordem real dos BLOCOS no circuito, obtida da primeira instrução parametrizada.
ORIGINAL_BLOCK_ORDER = parameter_map_df.sort_values("first_instruction")["theta_index"].astype(int).tolist()
if sorted(ORIGINAL_BLOCK_ORDER) != list(range(N_PARAMETERS)):
    raise RuntimeError("A ordem lógica não contém exatamente os 30 blocos.")

block_position = {theta_index: position for position, theta_index in enumerate(ORIGINAL_BLOCK_ORDER)}

active_structure_df = parameter_map_df.loc[
    parameter_map_df["theta_index"].isin(ACTIVE_THETA_INDICES)
].copy()
active_structure_df["original_block_position"] = active_structure_df["theta_index"].map(block_position)
active_structure_df = active_structure_df.sort_values("original_block_position")

# Estado-base realmente preparado pelos X iniciais, na convenção de bitstring do Qiskit.
initial_bitstring_qiskit = "".join(
    "1" if q in initial_x_qubits else "0"
    for q in range(N_ASSETS - 1, -1, -1)
)

# O ótimo de referência é usado SOMENTE para interpretação posterior.
optimal_probability_by_bitstring = {
    bitstring: float(reference_probability[label_to_index[bitstring]])
    for bitstring in exact_qiskit_bitstrings
}
REFERENCE_OPTIMAL_BITSTRING = max(
    exact_qiskit_bitstrings,
    key=lambda bitstring: optimal_probability_by_bitstring[bitstring],
)


def differing_qubits(bitstring_a, bitstring_b):
    """Converte diferenças na string Qiskit de volta para índices físicos de qubit."""
    return tuple(
        N_ASSETS - 1 - position
        for position, (a, b) in enumerate(zip(bitstring_a, bitstring_b))
        if a != b
    )


def transition_qubits(initial, final):
    removed, added = [], []
    for position, (a, b) in enumerate(zip(initial, final)):
        qubit = N_ASSETS - 1 - position
        if a == "1" and b == "0":
            removed.append(qubit)
        elif a == "0" and b == "1":
            added.append(qubit)
    return tuple(sorted(removed)), tuple(sorted(added))


removed_qubits, added_qubits = transition_qubits(
    initial_bitstring_qiskit, REFERENCE_OPTIMAL_BITSTRING
)
changed_qubits = set(removed_qubits) | set(added_qubits)

active_structure_df["touches_changed_qubit"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & changed_qubits)
)
active_structure_df["touches_removed_excitation"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & set(removed_qubits))
)
active_structure_df["touches_added_excitation"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & set(added_qubits))
)

route_columns = [
    "theta_index", "original_block_position", "ansatz_gate_type",
    "logical_block_qubits", "logical_assets", "primitive_physical_type",
    "first_instruction", "last_instruction", "touches_changed_qubit",
    "touches_removed_excitation", "touches_added_excitation",
]
display(active_structure_df[route_columns].reset_index(drop=True))

print("Bitstring inicial :", initial_bitstring_qiskit)
print("Ótimo de referência:", REFERENCE_OPTIMAL_BITSTRING)
print("Excitações que saem dos qubits:", removed_qubits)
print("Excitações que entram nos qubits:", added_qubits)

active_structure_df[route_columns].to_csv(
    STRUCTURAL_TABLE_DIR / "active7_structural_map.csv", index=False
)


### Célula 23 — reconstrução de uma arquitetura por ordem de blocos

A função abaixo é a peça central do experimento.

Ela **não move instruções primitivas isoladas**. Para cada identidade `theta_index`, reconstrói o bloco lógico inteiro:

- `CY`: `CX → CRY(θ) → CX`;
- `CCY`: `CX → RY(θ) → CCX → RY(-θ) → CCX → CX`.

Assim, quando `theta_17` é deslocado, são deslocadas junto com ele todas as operações que definem o bloco originalmente associado a `theta_17`.

A auditoria ao final reconstrói a ordem original e exige fidelidade unitária com o `ansatz` já usado nas Partes anteriores.


In [ ]:
# ============================================================
# 23. RECONSTRUTOR DO CIRCUITO POR BLOCOS — COM AUDITORIA
# ============================================================

structure_by_theta = structure_df.set_index("theta_index")


def build_circuit_from_block_order(block_order):
    """Reconstrói o ansatz movendo blocos completos e preservando sua identidade theta_j."""
    block_order = list(map(int, block_order))
    if sorted(block_order) != list(range(N_PARAMETERS)):
        raise ValueError("block_order deve ser uma permutação dos 30 theta_index.")

    theta_objects = ParameterVector("theta_struct", N_PARAMETERS)
    qc = QuantumCircuit(N_ASSETS)

    # Preparação inicial é idêntica em TODAS as arquiteturas.
    for qubit in initial_x_qubits:
        qc.x(int(qubit))

    for theta_index in block_order:
        row = structure_by_theta.loc[int(theta_index)]
        i_value, l_value = int(row["i"]), int(row["l"])
        theta_parameter = theta_objects[int(theta_index)]

        # CX externo que pertence ao bloco lógico original.
        qc.cx(i_value, l_value)

        if row["ansatz_gate_type"] == "CY":
            # Equivale ao CY_parameterized original: controle=l, alvo=i.
            qc.cry(theta_parameter, l_value, i_value)
        elif row["ansatz_gate_type"] == "CCY":
            # Equivale exatamente ao interior do CCY_parameterized original.
            qc.ry(theta_parameter, i_value)
            qc.ccx(l_value, i_value + 1, i_value)
            qc.ry(-theta_parameter, i_value)
            qc.ccx(l_value, i_value + 1, i_value)
        else:
            raise ValueError(f"Tipo de bloco inesperado: {row['ansatz_gate_type']}")

        # Segundo CX externo fecha o MESMO bloco lógico.
        qc.cx(i_value, l_value)

    return qc, tuple(theta_objects)


def bind_structural_circuit(circuit, parameter_objects, theta_values):
    """Binding explícito por identidade; não depende da ordenação interna de circuit.parameters."""
    theta_values = np.asarray(theta_values, dtype=float)
    mapping = {
        parameter_objects[j]: float(theta_values[j])
        for j in range(N_PARAMETERS)
    }
    return circuit.assign_parameters(mapping, inplace=False)


# AUDITORIA CRÍTICA: reconstruir a ordem original precisa reproduzir o ansatz anterior.
reconstructed_original, reconstructed_parameters = build_circuit_from_block_order(
    ORIGINAL_BLOCK_ORDER
)
old_state = np.asarray(
    Statevector.from_instruction(ansatz.assign_parameters(theta_reference, inplace=False)).data,
    dtype=np.complex128,
)
new_state = np.asarray(
    Statevector.from_instruction(
        bind_structural_circuit(reconstructed_original, reconstructed_parameters, theta_reference)
    ).data,
    dtype=np.complex128,
)
reconstruction_fidelity = float(abs(np.vdot(old_state, new_state)) ** 2)

if (1.0 - reconstruction_fidelity) > 1e-10:
    raise RuntimeError(
        "A reconstrução por blocos não reproduziu o circuito original. "
        f"Fidelidade={reconstruction_fidelity:.16f}"
    )

print(f"Fidelidade reconstrução/original = {reconstruction_fidelity:.16f}")
print("Auditoria aprovada: agora os blocos podem ser deslocados como unidades completas.")


### Célula 24 — medição e reotimização sem usar o bitstring como alvo

A energia é calculada a partir do **mesmo operador Ising**. Como esse Hamiltoniano é diagonal, sua energia em cada estado-base é pré-calculada uma única vez para acelerar as muitas reotimizações.

`optimize_block_order` recebe apenas:

1. uma ordem dos 30 blocos;
2. um vetor inicial;
3. os índices que podem variar.

Nos quatro testes principais, **todos os 30 parâmetros ficam livres**. Isso é importante: depois de mudar a arquitetura, um dos antigos 23 pode deixar de ser redundante. O experimento não força a conclusão anterior a permanecer verdadeira.


In [ ]:
# ============================================================
# 24. MEDIÇÃO + REOTIMIZAÇÃO DA ENERGIA
# ============================================================

# Energia de cada estado da base computacional. Isto acelera E=<H> sem mudar H.
try:
    _ising_matrix = ising.to_matrix(sparse=True)
    BASIS_ENERGIES = np.real(np.asarray(_ising_matrix.diagonal()).ravel()) + ising_offset
except TypeError:
    BASIS_ENERGIES = np.real(np.diag(np.asarray(ising.to_matrix()))) + ising_offset

valid_rank_lookup = {
    str(row.bitstring_qiskit_order): int(rank)
    for rank, row in enumerate(enumeration_df.itertuples(index=False), start=1)
}
valid_energy_lookup = {
    str(row.bitstring_qiskit_order): float(row.objective)
    for row in enumeration_df.itertuples(index=False)
}


def nearest_optimal_hamming(bitstring):
    return min(
        sum(a != b for a, b in zip(bitstring, optimum))
        for optimum in exact_qiskit_bitstrings
    )


def measure_structural_circuit(circuit, parameter_objects, theta_values, keep_probabilities=False):
    bound = bind_structural_circuit(circuit, parameter_objects, theta_values)
    state = Statevector.from_instruction(bound)
    probabilities = np.asarray(state.probabilities(), dtype=float)
    dominant_index = int(np.argmax(probabilities))
    dominant_bitstring = str(all_labels[dominant_index])
    expected_energy = float(np.dot(probabilities, BASIS_ENERGIES))

    result = {
        "expected_energy": expected_energy,
        "energy_gap": float(expected_energy - exact_energy),
        "p_optimal": float(probabilities[optimal_indices].sum()),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probabilities[dominant_index]),
        "dominant_is_exact_optimum": dominant_bitstring in set(exact_qiskit_bitstrings),
        "nearest_optimal_hamming": int(nearest_optimal_hamming(dominant_bitstring)),
        "dominant_valid_rank": valid_rank_lookup.get(dominant_bitstring, np.nan),
        "dominant_classical_energy": valid_energy_lookup.get(dominant_bitstring, np.nan),
    }
    if keep_probabilities:
        result["probabilities"] = probabilities
        result["statevector"] = np.asarray(state.data, dtype=np.complex128)
    return result


def optimize_block_order(
    label,
    block_order,
    theta_start,
    free_indices=None,
    maxiter=ARCHITECTURE_MAXITER,
    n_restarts=ARCHITECTURE_RESTARTS,
    seed=RANDOM_SEED,
):
    """Minimiza SOMENTE a energia. O bitstring ótimo nunca entra na função objetivo."""
    circuit, parameter_objects = build_circuit_from_block_order(block_order)
    free_indices = np.asarray(
        list(range(N_PARAMETERS)) if free_indices is None else list(free_indices),
        dtype=int,
    )
    theta_start = np.asarray(theta_start, dtype=float).copy()
    periods = parameter_map_df.set_index("theta_index")["angular_period"].to_dict()
    rng_local = np.random.default_rng(int(seed))

    starts = [theta_start.copy()]
    for _ in range(max(0, int(n_restarts) - 1)):
        candidate = theta_start.copy()
        candidate[free_indices] += np.asarray([
            rng_local.uniform(-0.20, 0.20) * periods[int(j)]
            for j in free_indices
        ])
        starts.append(candidate)

    best_theta = theta_start.copy()
    best_metrics = measure_structural_circuit(circuit, parameter_objects, best_theta)
    total_nfev = 0
    optimizer_messages = []

    for restart_id, start in enumerate(starts):
        template = start.copy()

        def energy_objective(free_values):
            theta_trial = template.copy()
            theta_trial[free_indices] = np.asarray(free_values, dtype=float)
            return measure_structural_circuit(
                circuit, parameter_objects, theta_trial
            )["expected_energy"]

        result = minimize(
            energy_objective,
            x0=start[free_indices],
            method="COBYLA",
            options={"maxiter": int(maxiter), "rhobeg": 0.5, "catol": 1e-10},
        )
        total_nfev += int(getattr(result, "nfev", 0))
        optimizer_messages.append(str(getattr(result, "message", "")))

        theta_candidate = template.copy()
        theta_candidate[free_indices] = np.asarray(result.x, dtype=float)
        metrics_candidate = measure_structural_circuit(
            circuit, parameter_objects, theta_candidate
        )

        # Nunca aceita uma saída pior que o melhor ponto já conhecido.
        if metrics_candidate["expected_energy"] < best_metrics["expected_energy"]:
            best_theta = theta_candidate
            best_metrics = metrics_candidate

    return {
        "label": str(label),
        "block_order": tuple(map(int, block_order)),
        "theta_opt": best_theta,
        "metrics": best_metrics,
        "circuit": circuit,
        "parameter_objects": parameter_objects,
        "nfev_total": int(total_nfev),
        "optimizer_messages": tuple(optimizer_messages),
    }


### Célula 25 — quatro arquiteturas causais

Os quatro casos são:

- `original`: controle, mesma ordem da arquitetura usada anteriormente;
- `active_first`: os 7 blocos sensíveis vão para o começo;
- `active_last`: os 7 vão para o final;
- `active_reverse_in_place`: somente os 7 trocam de identidade entre as posições originalmente ocupadas por eles; os 23 restantes permanecem exatamente nos mesmos slots.

Em todos os casos o Hamiltoniano e a preparação inicial permanecem idênticos e os **30 parâmetros são reotimizados**.


In [ ]:
# ============================================================
# 25. REOTIMIZAÇÃO: ORIGINAL / 7 NO INÍCIO / 7 NO FIM / 7 INVERTIDOS
# ============================================================

active_order_original = [j for j in ORIGINAL_BLOCK_ORDER if j in ACTIVE_THETA_INDICES]
inactive_order_original = [j for j in ORIGINAL_BLOCK_ORDER if j not in ACTIVE_THETA_INDICES]

order_active_first = active_order_original + inactive_order_original
order_active_last = inactive_order_original + active_order_original

# Os 23 ficam nos mesmos slots; somente os IDs dos 7 são invertidos entre si.
order_active_reverse = ORIGINAL_BLOCK_ORDER.copy()
active_slots = [position for position, j in enumerate(ORIGINAL_BLOCK_ORDER) if j in ACTIVE_THETA_INDICES]
for slot, new_theta_id in zip(active_slots, active_order_original[::-1]):
    order_active_reverse[slot] = int(new_theta_id)

ARCHITECTURE_ORDERS = {
    "original": ORIGINAL_BLOCK_ORDER,
    "active_first": order_active_first,
    "active_last": order_active_last,
    "active_reverse_in_place": order_active_reverse,
}

architecture_runs = {}
for run_id, (label, order) in enumerate(ARCHITECTURE_ORDERS.items()):
    print(f"Otimizando arquitetura: {label}")
    architecture_runs[label] = optimize_block_order(
        label=label,
        block_order=order,
        theta_start=theta_reference,
        free_indices=range(N_PARAMETERS),  # IMPORTANTE: todos os 30 ficam livres.
        maxiter=ARCHITECTURE_MAXITER,
        n_restarts=ARCHITECTURE_RESTARTS,
        seed=RANDOM_SEED + 1000 * run_id,
    )

architecture_rows = []
architecture_theta_rows = []
for label, run in architecture_runs.items():
    row = {"architecture": label, "nfev_total": run["nfev_total"], **run["metrics"]}
    architecture_rows.append(row)
    for theta_index, value in enumerate(run["theta_opt"]):
        architecture_theta_rows.append({
            "architecture": label,
            "theta_index": int(theta_index),
            "optimized_theta": float(value),
            "originally_sensitive": bool(theta_index in ACTIVE_THETA_INDICES),
            "block_position": int(run["block_order"].index(theta_index)),
        })

architecture_summary_df = pd.DataFrame(architecture_rows)
architecture_theta_long_df = pd.DataFrame(architecture_theta_rows)

display(architecture_summary_df[[
    "architecture", "expected_energy", "energy_gap", "p_optimal",
    "dominant_bitstring", "dominant_probability", "dominant_is_exact_optimum",
    "nearest_optimal_hamming", "dominant_valid_rank", "nfev_total",
]])

architecture_summary_df.to_csv(
    STRUCTURAL_TABLE_DIR / "architecture_reoptimization_summary.csv", index=False
)
architecture_theta_long_df.to_csv(
    STRUCTURAL_TABLE_DIR / "architecture_optimized_theta_long.csv", index=False
)


### Célula 26 — a sensibilidade permaneceu nos mesmos 7?

Depois de cada reotimização, cada um dos 30 parâmetros recebe duas perturbações locais estruturais:

\[
\theta_j\rightarrow\theta_j\pm T_j/4,
\]

onde \(T_j=2\pi\) para os blocos `RY/CCY` e \(T_j=4\pi\) para `CRY/CY`.

A classificação usa simultaneamente:

- mudança em \(P(\mathcal X_{opt})\);
- mudança de energia;
- TVD da distribuição completa;
- troca do bitstring dominante.

Isso permite verificar se, depois de mover os blocos, a sensibilidade **continua nos sete originais ou migra para algum dos antigos 23**.


In [ ]:
# ============================================================
# 26. PROBE DE SENSIBILIDADE DOS 30 PARÂMETROS EM CADA ARQUITETURA
# ============================================================

period_by_theta = parameter_map_df.set_index("theta_index")["angular_period"].to_dict()


def sensitivity_probe(run):
    base_theta = np.asarray(run["theta_opt"], dtype=float)
    circuit = run["circuit"]
    parameter_objects = run["parameter_objects"]
    base = measure_structural_circuit(
        circuit, parameter_objects, base_theta, keep_probabilities=True
    )
    rows = []

    for theta_index in range(N_PARAMETERS):
        shift = float(period_by_theta[theta_index] / 4.0)
        probe_metrics = []
        for sign in (-1.0, +1.0):
            trial = base_theta.copy()
            trial[theta_index] += sign * shift
            measured = measure_structural_circuit(
                circuit, parameter_objects, trial, keep_probabilities=True
            )
            measured["tvd"] = float(
                0.5 * np.abs(measured["probabilities"] - base["probabilities"]).sum()
            )
            probe_metrics.append(measured)

        max_dp = max(abs(m["p_optimal"] - base["p_optimal"]) for m in probe_metrics)
        max_de = max(abs(m["expected_energy"] - base["expected_energy"]) for m in probe_metrics)
        max_tvd = max(m["tvd"] for m in probe_metrics)
        bit_changed = any(
            m["dominant_bitstring"] != base["dominant_bitstring"]
            for m in probe_metrics
        )
        sensitive = bool(
            max_dp > SENSITIVITY_PROB_TOL
            or max_de > SENSITIVITY_ENERGY_TOL
            or max_tvd > SENSITIVITY_TVD_TOL
            or bit_changed
        )
        rows.append({
            "theta_index": int(theta_index),
            "block_position": int(run["block_order"].index(theta_index)),
            "originally_sensitive": bool(theta_index in ACTIVE_THETA_INDICES),
            "probe_shift": shift,
            "max_abs_delta_p_optimal": float(max_dp),
            "max_abs_delta_energy": float(max_de),
            "max_tvd": float(max_tvd),
            "dominant_bitstring_changed": bool(bit_changed),
            "sensitive_after_reorder": sensitive,
        })
    return pd.DataFrame(rows)


sensitivity_frames = []
for architecture, run in architecture_runs.items():
    frame = sensitivity_probe(run)
    frame.insert(0, "architecture", architecture)
    sensitivity_frames.append(frame)

sensitivity_by_architecture_df = pd.concat(sensitivity_frames, ignore_index=True)

sensitivity_by_architecture_df["original7_and_sensitive"] = (
    sensitivity_by_architecture_df["originally_sensitive"]
    & sensitivity_by_architecture_df["sensitive_after_reorder"]
)

sensitivity_summary_df = (
    sensitivity_by_architecture_df
    .groupby("architecture", as_index=False)
    .agg(
        n_sensitive_after_reorder=("sensitive_after_reorder", "sum"),
        n_original7_still_sensitive=("original7_and_sensitive", "sum"),
    )
)

sensitive_only_df = sensitivity_by_architecture_df.loc[
    sensitivity_by_architecture_df["sensitive_after_reorder"]
].sort_values(["architecture", "block_position"])

display(sensitivity_summary_df)
display(sensitive_only_df)

sensitivity_by_architecture_df.to_csv(
    STRUCTURAL_TABLE_DIR / "sensitivity_after_each_architecture.csv", index=False
)


### Célula 27 — deslocamento individual dos 7 blocos por todas as posições

Este é o teste de posição mais direto.

Para cada um dos 7 blocos:

1. remove o bloco da posição original;
2. insere o **mesmo bloco completo** em cada posição de 0 a 29;
3. mantém a ordem relativa dos outros 29 blocos;
4. reotimiza **todos os 30 parâmetros**;
5. mede energia, \(P_{opt}\), bitstring dominante e distância de Hamming;
6. registra também os valores otimizados dos **sete parâmetros originais**.

O ponto da posição original é reutilizado do controle já otimizado, evitando sete reotimizações redundantes.

`RUN_FULL_POSITION_SCAN=False` pode ser usado para pular esta etapa durante uma checagem rápida das células anteriores.


In [ ]:
# ============================================================
# 27. VARREDURA CAUSAL DE POSIÇÃO DOS 7 BLOCOS
# ============================================================


def move_one_block(base_order, theta_index, target_position):
    order = list(map(int, base_order))
    order.remove(int(theta_index))
    order.insert(int(target_position), int(theta_index))
    return order


position_scan_rows = []
position_theta_rows = []

if RUN_FULL_POSITION_SCAN:
    original_control = architecture_runs["original"]
    theta_start_scan = np.asarray(original_control["theta_opt"], dtype=float)

    for moved_theta in ACTIVE_THETA_INDICES:
        original_position = int(ORIGINAL_BLOCK_ORDER.index(int(moved_theta)))
        print(f"Varredura de posição: theta_{moved_theta} (posição original {original_position})")

        for target_position in range(N_PARAMETERS):
            if target_position == original_position:
                run = original_control
            else:
                order = move_one_block(
                    ORIGINAL_BLOCK_ORDER, moved_theta, target_position
                )
                run = optimize_block_order(
                    label=f"theta_{moved_theta}_to_{target_position}",
                    block_order=order,
                    theta_start=theta_start_scan,
                    free_indices=range(N_PARAMETERS),
                    maxiter=POSITION_SCAN_MAXITER,
                    n_restarts=POSITION_SCAN_RESTARTS,
                    seed=RANDOM_SEED + 10000 + 100 * int(moved_theta) + target_position,
                )

            metrics = run["metrics"]
            position_scan_rows.append({
                "moved_theta": int(moved_theta),
                "original_position": original_position,
                "target_position": int(target_position),
                "position_shift": int(target_position - original_position),
                "expected_energy": metrics["expected_energy"],
                "energy_gap": metrics["energy_gap"],
                "p_optimal": metrics["p_optimal"],
                "dominant_bitstring": metrics["dominant_bitstring"],
                "dominant_probability": metrics["dominant_probability"],
                "dominant_is_exact_optimum": metrics["dominant_is_exact_optimum"],
                "nearest_optimal_hamming": metrics["nearest_optimal_hamming"],
                "dominant_valid_rank": metrics["dominant_valid_rank"],
                "nfev_total": run["nfev_total"],
            })

            # Long format: mostra como TODOS os 7 theta respondem ao deslocamento de um deles.
            for tracked_theta in ACTIVE_THETA_INDICES:
                position_theta_rows.append({
                    "moved_theta": int(moved_theta),
                    "target_position": int(target_position),
                    "tracked_theta": int(tracked_theta),
                    "optimized_theta": float(run["theta_opt"][int(tracked_theta)]),
                })

    position_scan_df = pd.DataFrame(position_scan_rows)
    position_theta_trace_df = pd.DataFrame(position_theta_rows)

    position_scan_summary_df = (
        position_scan_df.groupby("moved_theta", as_index=False)
        .agg(
            original_position=("original_position", "first"),
            min_energy_gap=("energy_gap", "min"),
            max_energy_gap=("energy_gap", "max"),
            min_p_optimal=("p_optimal", "min"),
            max_p_optimal=("p_optimal", "max"),
            n_positions_exact_dominant=("dominant_is_exact_optimum", "sum"),
            max_hamming=("nearest_optimal_hamming", "max"),
        )
    )

    display(position_scan_summary_df)

    changed_bitstrings_df = position_scan_df.loc[
        ~position_scan_df["dominant_is_exact_optimum"]
    ].sort_values(["moved_theta", "target_position"])
    print("Posições em que o bitstring dominante deixou de ser ótimo:", len(changed_bitstrings_df))
    display(changed_bitstrings_df.head(40))

    position_scan_df.to_csv(
        STRUCTURAL_TABLE_DIR / "active7_full_position_scan.csv", index=False
    )
    position_theta_trace_df.to_csv(
        STRUCTURAL_TABLE_DIR / "active7_theta_response_during_position_scan.csv", index=False
    )
else:
    print("Varredura completa de posição pulada por RUN_FULL_POSITION_SCAN=False.")


### Célula 28 — visualização mínima e critérios de interpretação

Os mapas abaixo são deliberadamente simples:

- linha = identidade do bloco sensível movido;
- coluna = nova posição do bloco;
- primeiro mapa = probabilidade total dos bitstrings ótimos;
- segundo mapa = distância de Hamming do bitstring dominante até o ótimo mais próximo.

A interpretação é causal:

- **linha quase constante**: aquele bloco tolera deslocamento e sua identidade é mais importante que a posição;
- **faixa estreita de posições boas**: a posição é parte do mecanismo;
- **troca de sensibilidade para antigos 23** na Célula 26: a arquitetura, e não o índice original, determina quais parâmetros controlam;
- **mudança de bitstring com energia quase inalterada**: verificar `dominant_valid_rank` e `energy_gap` antes de interpretar como degenerescência;
- **os mesmos 7 continuam sensíveis em todas as ordens**: evidência favorável a uma subestrutura funcional associada aos qubits/blocos desses sete.


In [ ]:
# ============================================================
# 28. MAPAS MÍNIMOS: P(ÓTIMO) E DISTÂNCIA DE HAMMING
# ============================================================

if RUN_FULL_POSITION_SCAN and not position_scan_df.empty:
    p_matrix = position_scan_df.pivot(
        index="moved_theta", columns="target_position", values="p_optimal"
    ).sort_index()

    fig, ax = plt.subplots(figsize=(12, 4.5))
    image = ax.imshow(p_matrix.to_numpy(), aspect="auto", origin="lower")
    ax.set_yticks(range(len(p_matrix.index)))
    ax.set_yticklabels([f"theta_{j}" for j in p_matrix.index])
    ax.set_xticks(range(N_PARAMETERS))
    ax.set_xticklabels(range(N_PARAMETERS), rotation=90)
    ax.set_xlabel("Nova posição do bloco")
    ax.set_ylabel("Bloco deslocado")
    ax.set_title("P(bitstring ótimo) após reotimização")
    fig.colorbar(image, ax=ax, label="P_optimal")
    fig.tight_layout()
    fig.savefig(STRUCTURAL_FIGURE_DIR / "position_scan_p_optimal.png", dpi=180)
    plt.show()

    hamming_matrix = position_scan_df.pivot(
        index="moved_theta", columns="target_position", values="nearest_optimal_hamming"
    ).sort_index()

    fig, ax = plt.subplots(figsize=(12, 4.5))
    image = ax.imshow(hamming_matrix.to_numpy(), aspect="auto", origin="lower")
    ax.set_yticks(range(len(hamming_matrix.index)))
    ax.set_yticklabels([f"theta_{j}" for j in hamming_matrix.index])
    ax.set_xticks(range(N_PARAMETERS))
    ax.set_xticklabels(range(N_PARAMETERS), rotation=90)
    ax.set_xlabel("Nova posição do bloco")
    ax.set_ylabel("Bloco deslocado")
    ax.set_title("Distância de Hamming do dominante ao ótimo mais próximo")
    fig.colorbar(image, ax=ax, label="Hamming")
    fig.tight_layout()
    fig.savefig(STRUCTURAL_FIGURE_DIR / "position_scan_hamming.png", dpi=180)
    plt.show()


# Parte VIII — ação variacional, geometria do ansatz e soma sobre caminhos

## O que esta parte testa

Os experimentos anteriores mostraram empiricamente que 23 parâmetros podem ser alterados sem modificar a solução, enquanto 7 parâmetros aparecem como sensíveis. Agora queremos saber **qual mecanismo matemático produz essa redução efetiva**.

Para um ansatz parametrizado

\[
|\psi(\boldsymbol\theta)\rangle,
\]

cada parâmetro define uma direção tangente

\[
|\partial_i\psi\rangle=\frac{\partial|\psi\rangle}{\partial\theta_i}.
\]

A métrica de Fubini–Study é obtida do tensor geométrico quântico:

\[
g_{ij}=\operatorname{Re}\left[
\langle\partial_i\psi|\partial_j\psi\rangle-
\langle\partial_i\psi|\psi\rangle
\langle\psi|\partial_j\psi\rangle
\right].
\]

A diagonal \(g_{ii}\) mede quanto o **estado físico**, retirando fase global, se move quando \(\theta_i\) varia.

### Um cuidado importante no ponto ótimo

Em um mínimo variacional esperamos

\[
\frac{\partial E}{\partial\theta_i}\approx0
\]

para **todos** os parâmetros. Portanto o gradiente de energia não deve ser usado sozinho para decidir quais θ são importantes. O diagnóstico de rigidez local será a curvatura

\[
\frac{\partial^2 E}{\partial\theta_i^2}.
\]

Assim conseguimos separar:

- **direção geométrica nula:** \(g_{ii}\approx0\);
- **direção energeticamente plana:** \(\partial_i^2E\approx0\);
- **direção que muda o estado mas não a energia:** \(g_{ii}>0\) e curvatura energética pequena.

A segunda metade desta parte trata o circuito como uma soma discreta sobre histórias. A amplitude final

\[
\langle x_f|U_L\cdots U_1|x_i\rangle
\]

é propagada bloco a bloco, somando coerentemente as contribuições de todos os estados intermediários. Isso é o análogo discreto, no circuito, da lógica de soma sobre caminhos de Feynman.


### Célula 29 — referência física e parâmetros numéricos da Parte VIII

A referência desta etapa é a arquitetura `original` **reotimizada com os 30 θ livres** na Parte VII. Isso evita usar os máximos 1D como se fossem um ótimo conjunto.

O bitstring ótimo continua sendo apenas um **observável posterior**. Ele nunca entra na função objetivo do COBYLA.

Os passos finitos abaixo são pequenos e servem somente para derivadas numéricas do `Statevector`. As tolerâncias de classificação são relativas ao maior sinal observado, para que a tabela mostre também os valores brutos usados na decisão.


In [ ]:
# ============================================================
# 29. CONFIGURAÇÃO — GEOMETRIA VARIACIONAL E PATH-SUM
# ============================================================

from collections import defaultdict
from qiskit.quantum_info import Operator

ACTION_ROOT = OUTPUT_ROOT / "action_variational_pathsum"
ACTION_TABLE_DIR = ACTION_ROOT / "tables"
ACTION_FIGURE_DIR = ACTION_ROOT / "figures"
for directory in [ACTION_ROOT, ACTION_TABLE_DIR, ACTION_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Passo central para derivadas do estado e curvaturas locais.
FD_STEP = 1e-4
QGT_RELATIVE_NULL_TOL = 1e-6
CURVATURE_RELATIVE_NULL_TOL = 1e-6
AMPLITUDE_RELATIVE_NULL_TOL = 1e-6

# Path-sum: apenas zeros numéricos são descartados.
PATH_MATRIX_ELEMENT_TOL = 1e-14
PATH_AMPLITUDE_TOL = 1e-14
TOP_STATES_PER_LAYER = 12

# Trajetória geométrica opcional. Não é chamada de geodésica.
RUN_PARAMETER_PATH_GEOMETRY = True
PARAMETER_PATH_POINTS = 9

# Referência física: circuito ORIGINAL após reotimização conjunta dos 30 parâmetros.
action_run = architecture_runs["original"]
ACTION_CIRCUIT = action_run["circuit"]
ACTION_PARAMETERS = action_run["parameter_objects"]
ACTION_THETA = np.asarray(action_run["theta_opt"], dtype=float).copy()
action_metrics = measure_structural_circuit(
    ACTION_CIRCUIT, ACTION_PARAMETERS, ACTION_THETA, keep_probabilities=True
)
ACTION_STATE = np.asarray(action_metrics["statevector"], dtype=np.complex128)
ACTION_PROBABILITY = np.asarray(action_metrics["probabilities"], dtype=float)

# Se houver degenerescência clássica, escolhe para o path-sum o ótimo que recebe
# maior probabilidade NESTA referência reotimizada. Isso não altera a otimização.
PATH_TARGET_BITSTRING = max(
    exact_qiskit_bitstrings,
    key=lambda bit: float(ACTION_PROBABILITY[label_to_index[bit]]),
)
PATH_TARGET_INDEX = int(label_to_index[PATH_TARGET_BITSTRING])

print("Referência da Parte VIII: arquitetura original reotimizada")
print("Energia =", action_metrics["expected_energy"])
print("P(ótimos) =", action_metrics["p_optimal"])
print("Bitstring alvo apenas para diagnóstico =", PATH_TARGET_BITSTRING)
print("Saídas =", ACTION_ROOT.resolve())


### Célula 30 — QGT/Fubini–Study, gradiente e curvatura de energia

Esta célula faz somente uma operação conceitual: desloca cada θ por \(\pm\varepsilon\), calcula os dois `Statevector` e forma a derivada central

\[
|\partial_i\psi\rangle\approx
\frac{|\psi(\theta_i+\varepsilon)\rangle-|\psi(\theta_i-\varepsilon)\rangle}{2\varepsilon}.
\]

Com essas 30 derivadas, todo o QGT é construído por produtos internos. Na mesma avaliação são calculados:

- gradiente de energia — deve ficar pequeno no mínimo;
- **curvatura de energia** — indica rigidez ou planicidade;
- derivada e curvatura da probabilidade do bitstring ótimo escolhido;
- derivada e curvatura da probabilidade total do conjunto de ótimos.

Nenhuma nova otimização é executada.


In [ ]:
# ============================================================
# 30. DERIVADAS DO ESTADO + QGT + CURVATURAS
# ============================================================


def structural_state(theta_values):
    """Statevector do circuito estrutural original para um vetor theta."""
    bound = bind_structural_circuit(ACTION_CIRCUIT, ACTION_PARAMETERS, theta_values)
    return np.asarray(Statevector.from_instruction(bound).data, dtype=np.complex128)


def energy_from_state(state):
    """Como H é diagonal, E é o produto das probabilidades pelas energias da base."""
    return float(np.dot(np.abs(state) ** 2, BASIS_ENERGIES))


base_state = structural_state(ACTION_THETA)
base_energy = energy_from_state(base_state)
base_p_target = float(abs(base_state[PATH_TARGET_INDEX]) ** 2)
base_p_optimal = float(np.sum(np.abs(base_state[optimal_indices]) ** 2))

state_derivatives = []
finite_rows = []

for theta_index in range(N_PARAMETERS):
    plus = ACTION_THETA.copy()
    minus = ACTION_THETA.copy()
    plus[theta_index] += FD_STEP
    minus[theta_index] -= FD_STEP

    psi_plus = structural_state(plus)
    psi_minus = structural_state(minus)
    dpsi = (psi_plus - psi_minus) / (2.0 * FD_STEP)
    state_derivatives.append(dpsi)

    e_plus, e_minus = energy_from_state(psi_plus), energy_from_state(psi_minus)
    p_target_plus = float(abs(psi_plus[PATH_TARGET_INDEX]) ** 2)
    p_target_minus = float(abs(psi_minus[PATH_TARGET_INDEX]) ** 2)
    p_opt_plus = float(np.sum(np.abs(psi_plus[optimal_indices]) ** 2))
    p_opt_minus = float(np.sum(np.abs(psi_minus[optimal_indices]) ** 2))

    finite_rows.append({
        "theta_index": int(theta_index),
        "energy_gradient_fd": float((e_plus - e_minus) / (2.0 * FD_STEP)),
        "energy_curvature_fd": float((e_plus - 2.0 * base_energy + e_minus) / FD_STEP**2),
        "p_target_gradient_fd": float((p_target_plus - p_target_minus) / (2.0 * FD_STEP)),
        "p_target_curvature_fd": float((p_target_plus - 2.0 * base_p_target + p_target_minus) / FD_STEP**2),
        "p_optimal_gradient_fd": float((p_opt_plus - p_opt_minus) / (2.0 * FD_STEP)),
        "p_optimal_curvature_fd": float((p_opt_plus - 2.0 * base_p_optimal + p_opt_minus) / FD_STEP**2),
    })

DPSI = np.asarray(state_derivatives, dtype=np.complex128)

# <d_i|d_j> e <d_i|psi>; a segunda parcela remove a direção de fase global.
gram = DPSI.conj() @ DPSI.T
overlap_with_state = DPSI.conj() @ base_state
QGT_METRIC = np.real(
    gram - np.outer(overlap_with_state, np.conj(overlap_with_state))
)
QGT_METRIC = 0.5 * (QGT_METRIC + QGT_METRIC.T)

qgt_diag = np.clip(np.diag(QGT_METRIC), 0.0, None)
raw_derivative_norm = np.linalg.norm(DPSI, axis=1)
tangent_norm = np.sqrt(qgt_diag)

finite_difference_df = pd.DataFrame(finite_rows)
variational_geometry_df = parameter_map_df.copy().merge(
    finite_difference_df, on="theta_index", how="left", validate="one_to_one"
)
variational_geometry_df["qgt_diag"] = qgt_diag
variational_geometry_df["raw_state_derivative_norm"] = raw_derivative_norm
variational_geometry_df["fubini_study_tangent_norm"] = tangent_norm
variational_geometry_df["originally_sensitive"] = variational_geometry_df["theta_index"].isin(
    ACTIVE_THETA_INDICES
)

variational_geometry_df.to_csv(
    ACTION_TABLE_DIR / "variational_geometry_qgt_curvature_30theta.csv", index=False
)
display(variational_geometry_df[[
    "theta_index", "originally_sensitive", "ansatz_gate_type", "logical_block_qubits",
    "qgt_diag", "fubini_study_tangent_norm", "energy_gradient_fd",
    "energy_curvature_fd", "p_optimal_curvature_fd",
]].sort_values("qgt_diag", ascending=False))


### Célula 31 — espectro do QGT e dimensão efetiva

Se a família nominal de 30 parâmetros realmente se comporta como uma variedade de dimensão muito menor perto da solução, o espectro de \(g\) deve apresentar poucos autovalores relevantes.

Dois números são registrados:

1. **posto numérico:** quantidade de autovalores acima de uma tolerância relativa;
2. **dimensão de participação:**

\[
d_{\mathrm{PR}}=\frac{(\sum_a\lambda_a)^2}{\sum_a\lambda_a^2}.
\]

Também calculamos qual fração da diagonal do QGT está concentrada nos 7 θ já identificados empiricamente. Essa fração é um diagnóstico, não uma imposição — os 7 não entram no cálculo de \(g\).


In [ ]:
# ============================================================
# 31. ESPECTRO DO QGT E DIMENSÃO EFETIVA
# ============================================================

qgt_eigenvalues = np.linalg.eigvalsh(QGT_METRIC)
qgt_eigenvalues = np.clip(qgt_eigenvalues, 0.0, None)[::-1]
max_eigenvalue = float(qgt_eigenvalues[0]) if len(qgt_eigenvalues) else 0.0
rank_threshold = max(1e-14, QGT_RELATIVE_NULL_TOL * max_eigenvalue)
qgt_numeric_rank = int(np.sum(qgt_eigenvalues > rank_threshold))

if np.sum(qgt_eigenvalues**2) > 0:
    qgt_participation_dimension = float(
        np.sum(qgt_eigenvalues) ** 2 / np.sum(qgt_eigenvalues**2)
    )
else:
    qgt_participation_dimension = 0.0

active_mask = np.array([j in ACTIVE_THETA_INDICES for j in range(N_PARAMETERS)], dtype=bool)
metric_trace = float(np.sum(qgt_diag))
active_metric_fraction = float(np.sum(qgt_diag[active_mask]) / metric_trace) if metric_trace > 0 else np.nan

qgt_spectrum_df = pd.DataFrame({
    "eigenvalue_rank": np.arange(1, N_PARAMETERS + 1, dtype=int),
    "qgt_eigenvalue": qgt_eigenvalues,
    "above_numeric_rank_threshold": qgt_eigenvalues > rank_threshold,
})
qgt_summary_df = pd.DataFrame([{
    "n_nominal_parameters": int(N_PARAMETERS),
    "qgt_numeric_rank": qgt_numeric_rank,
    "qgt_participation_dimension": qgt_participation_dimension,
    "rank_threshold": rank_threshold,
    "active7_metric_trace_fraction": active_metric_fraction,
}])

# Classificação relativa; os valores brutos continuam na tabela para auditoria.
metric_null_threshold = max(1e-14, QGT_RELATIVE_NULL_TOL * float(np.max(qgt_diag)))
curvature_scale = float(np.max(np.abs(variational_geometry_df["energy_curvature_fd"])))
curvature_null_threshold = max(1e-12, CURVATURE_RELATIVE_NULL_TOL * curvature_scale)

variational_geometry_df["geometrically_null"] = variational_geometry_df["qgt_diag"] <= metric_null_threshold
variational_geometry_df["energetically_flat"] = (
    variational_geometry_df["energy_curvature_fd"].abs() <= curvature_null_threshold
)

print("Posto numérico do QGT =", qgt_numeric_rank, "/", N_PARAMETERS)
print("Dimensão de participação =", qgt_participation_dimension)
print("Fração Tr(g) nos 7 theta =", active_metric_fraction)
display(qgt_summary_df)

qgt_spectrum_df.to_csv(ACTION_TABLE_DIR / "qgt_eigenvalue_spectrum.csv", index=False)
qgt_summary_df.to_csv(ACTION_TABLE_DIR / "qgt_effective_dimension_summary.csv", index=False)
variational_geometry_df.to_csv(
    ACTION_TABLE_DIR / "variational_geometry_qgt_curvature_30theta.csv", index=False
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(qgt_spectrum_df["eigenvalue_rank"], np.maximum(qgt_eigenvalues, 1e-18), marker="o")
ax.axhline(rank_threshold, linestyle="--", linewidth=1.0, label="limiar de posto")
ax.set_xlabel("ordem do autovalor")
ax.set_ylabel("autovalor do QGT")
ax.set_title("Espectro da métrica de Fubini–Study no ponto otimizado")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "qgt_eigenvalue_spectrum.png", dpi=180)
plt.show()


### Célula 32 — influência de cada θ na amplitude do ótimo

Para o bitstring ótimo escolhido apenas como diagnóstico,

\[
A_f(\boldsymbol\theta)=\langle x_f|\psi(\boldsymbol\theta)\rangle,
\]

calculamos

\[
\frac{\partial A_f}{\partial\theta_i}
=\langle x_f|\partial_i\psi\rangle.
\]

Essa quantidade é equivalente à forma “forward/backward”

\[
\left\langle B_i\left|\frac{\partial U_i}{\partial\theta_i}\right|F_{i-1}\right\rangle,
\]

mas aqui é obtida diretamente do `Statevector` derivado, evitando reconstruções adicionais e mantendo a auditoria simples.

Como a probabilidade ótima pode estar próxima de 1, sua primeira derivada também pode zerar no máximo. Por isso guardamos simultaneamente **derivada de amplitude**, QGT e **curvatura de probabilidade**.


In [ ]:
# ============================================================
# 32. DERIVADA DA AMPLITUDE DO BITSTRING ÓTIMO
# ============================================================

reference_amplitude = complex(base_state[PATH_TARGET_INDEX])
d_amplitude = DPSI[:, PATH_TARGET_INDEX]

# Derivada da probabilidade de UM ótimo e da massa em TODOS os ótimos.
d_probability_target = 2.0 * np.real(np.conj(reference_amplitude) * d_amplitude)
d_probability_all_optima = 2.0 * np.real(
    np.sum(np.conj(base_state[optimal_indices])[None, :] * DPSI[:, optimal_indices], axis=1)
)

amplitude_influence_df = pd.DataFrame({
    "theta_index": np.arange(N_PARAMETERS, dtype=int),
    "originally_sensitive": [j in ACTIVE_THETA_INDICES for j in range(N_PARAMETERS)],
    "abs_d_amplitude_target": np.abs(d_amplitude),
    "real_d_amplitude_target": np.real(d_amplitude),
    "imag_d_amplitude_target": np.imag(d_amplitude),
    "d_probability_target": d_probability_target,
    "d_probability_all_optima": d_probability_all_optima,
})

amp_scale = float(amplitude_influence_df["abs_d_amplitude_target"].max())
amp_null_threshold = max(1e-14, AMPLITUDE_RELATIVE_NULL_TOL * amp_scale)
amplitude_influence_df["amplitude_decoupled"] = (
    amplitude_influence_df["abs_d_amplitude_target"] <= amp_null_threshold
)

amplitude_influence_df = amplitude_influence_df.merge(
    parameter_map_df[[
        "theta_index", "ansatz_gate_type", "logical_block_qubits", "logical_assets",
        "first_instruction", "last_instruction",
    ]],
    on="theta_index", how="left", validate="one_to_one"
)

amplitude_influence_df.to_csv(
    ACTION_TABLE_DIR / "target_amplitude_influence_30theta.csv", index=False
)
display(amplitude_influence_df.sort_values("abs_d_amplitude_target", ascending=False))


### Célula 33 — propagação bloco a bloco

Agora o circuito é observado como uma sequência de 30 transformações lógicas.

Começamos exatamente no estado preparado pelas portas `X` e, após cada bloco, registramos:

- \(P(x_f)\) para o ótimo escolhido;
- probabilidade total dos ótimos degenerados;
- bitstring dominante;
- TVD em relação à camada anterior;
- quantidade de estados com probabilidade numericamente relevante;
- os bitstrings mais prováveis e suas fases.

A última linha deve reproduzir o `Statevector` completo da referência. Essa fidelidade é uma auditoria obrigatória antes da interpretação do caminho.


In [ ]:
# ============================================================
# 33. PROPAGAÇÃO CAMADA A CAMADA
# ============================================================


def bound_logical_block(theta_index, theta_value):
    """Constrói somente UM bloco lógico, já com o valor numérico de theta."""
    row = structure_by_theta.loc[int(theta_index)]
    i_value, l_value = int(row["i"]), int(row["l"])
    qc = QuantumCircuit(N_ASSETS)
    qc.cx(i_value, l_value)
    if row["ansatz_gate_type"] == "CY":
        qc.cry(float(theta_value), l_value, i_value)
    else:
        qc.ry(float(theta_value), i_value)
        qc.ccx(l_value, i_value + 1, i_value)
        qc.ry(-float(theta_value), i_value)
        qc.ccx(l_value, i_value + 1, i_value)
    qc.cx(i_value, l_value)
    return qc


initial_circuit = QuantumCircuit(N_ASSETS)
for qubit in initial_x_qubits:
    initial_circuit.x(int(qubit))
layer_state = Statevector.from_instruction(initial_circuit)

layer_rows = []
top_state_rows = []
previous_probability = np.asarray(layer_state.probabilities(), dtype=float)


def record_layer(step, theta_index, state, previous_probability):
    probability = np.asarray(state.probabilities(), dtype=float)
    dominant_index = int(np.argmax(probability))
    row = {
        "step": int(step),
        "theta_index": theta_index,
        "is_original7": bool(theta_index in ACTIVE_THETA_INDICES) if theta_index is not None else False,
        "p_target": float(probability[PATH_TARGET_INDEX]),
        "p_all_optima": float(probability[optimal_indices].sum()),
        "dominant_bitstring": str(all_labels[dominant_index]),
        "dominant_probability": float(probability[dominant_index]),
        "tvd_from_previous": float(0.5 * np.abs(probability - previous_probability).sum()),
        "n_states_probability_gt_1e_12": int(np.sum(probability > 1e-12)),
    }
    top_indices = np.argsort(probability)[::-1][:TOP_STATES_PER_LAYER]
    for rank, basis_index in enumerate(top_indices, start=1):
        amplitude = complex(state.data[int(basis_index)])
        top_state_rows.append({
            "step": int(step),
            "theta_index": theta_index,
            "rank": int(rank),
            "bitstring": str(all_labels[int(basis_index)]),
            "probability": float(probability[int(basis_index)]),
            "amplitude_abs": float(abs(amplitude)),
            "amplitude_phase": float(np.angle(amplitude)),
            "hamming_to_target": int(sum(
                a != b for a, b in zip(str(all_labels[int(basis_index)]), PATH_TARGET_BITSTRING)
            )),
        })
    return row, probability


row0, previous_probability = record_layer(0, None, layer_state, previous_probability)
layer_rows.append(row0)

for step, theta_index in enumerate(ORIGINAL_BLOCK_ORDER, start=1):
    block = bound_logical_block(theta_index, ACTION_THETA[int(theta_index)])
    layer_state = layer_state.evolve(block)
    row, current_probability = record_layer(step, int(theta_index), layer_state, previous_probability)
    row["delta_p_target"] = float(row["p_target"] - layer_rows[-1]["p_target"])
    row["delta_p_all_optima"] = float(row["p_all_optima"] - layer_rows[-1]["p_all_optima"])
    layer_rows.append(row)
    previous_probability = current_probability

layer_dynamics_df = pd.DataFrame(layer_rows)
layer_top_states_df = pd.DataFrame(top_state_rows)

layer_final_state = np.asarray(layer_state.data, dtype=np.complex128)
layer_final_fidelity = float(abs(np.vdot(ACTION_STATE, layer_final_state)) ** 2)
if (1.0 - layer_final_fidelity) > 1e-10:
    raise RuntimeError(f"Propagação camada a camada não reproduziu o circuito: F={layer_final_fidelity}")

print(f"Fidelidade final da propagação por blocos = {layer_final_fidelity:.16f}")
display(layer_dynamics_df)
layer_dynamics_df.to_csv(ACTION_TABLE_DIR / "layer_by_layer_dynamics.csv", index=False)
layer_top_states_df.to_csv(ACTION_TABLE_DIR / "top_basis_states_each_layer.csv", index=False)


### Célula 34 — soma discreta sobre caminhos e interferência

Esta é a implementação mais próxima da ideia de Feynman nesta arquitetura.

Cada bloco atua apenas em 2 ou 3 qubits. Para cada estado-base que possui amplitude na camada atual, calculamos todos os estados-base alcançáveis pelo bloco e somamos

\[
A_{l+1}(x')=\sum_x \langle x'|U_l|x\rangle A_l(x).
\]

A recursão acima é uma **soma exata e coerente sobre estados intermediários**; ela não amostra trajetórias.

Para cada estado de saída também podemos comparar

\[
\left|\sum_r c_r\right|^2
\quad\text{com}\quad
\sum_r|c_r|^2,
\]

onde \(c_r\) são as contribuições recebidas dos predecessores. A diferença mede interferência construtiva ou destrutiva naquela camada.

O código mantém as amplitudes complexas completas. O limiar só remove zeros de máquina.


In [ ]:
# ============================================================
# 34. PATH-SUM DISCRETO EXATO POR PROGRAMAÇÃO DINÂMICA
# ============================================================


def local_block_operator(theta_index, theta_value):
    """Retorna (qubits físicos, matriz local) do mesmo bloco usado no circuito."""
    row = structure_by_theta.loc[int(theta_index)]
    i_value, l_value = int(row["i"]), int(row["l"])

    if row["ansatz_gate_type"] == "CY":
        physical_qubits = (i_value, l_value)  # slots locais 0=i, 1=l
        qc = QuantumCircuit(2)
        qc.cx(0, 1)
        qc.cry(float(theta_value), 1, 0)
        qc.cx(0, 1)
    else:
        physical_qubits = (i_value, i_value + 1, l_value)  # slots 0=i,1=i+1,2=l
        qc = QuantumCircuit(3)
        qc.cx(0, 2)
        qc.ry(float(theta_value), 0)
        qc.ccx(2, 1, 0)
        qc.ry(-float(theta_value), 0)
        qc.ccx(2, 1, 0)
        qc.cx(0, 2)

    return physical_qubits, np.asarray(Operator(qc).data, dtype=np.complex128)


def replace_local_bits(global_index, physical_qubits, local_out_index):
    """Troca apenas os bits dos qubits tocados pelo bloco."""
    out_index = int(global_index)
    for local_slot, physical_qubit in enumerate(physical_qubits):
        bit = (int(local_out_index) >> local_slot) & 1
        if bit:
            out_index |= (1 << int(physical_qubit))
        else:
            out_index &= ~(1 << int(physical_qubit))
    return out_index


def pathsum_step(amplitudes, path_counts, theta_index, theta_value):
    """Propaga uma camada e soma coerentemente todas as contribuições recebidas."""
    physical_qubits, local_u = local_block_operator(theta_index, theta_value)
    next_amplitudes = defaultdict(complex)
    next_counts = defaultdict(int)
    contributions = defaultdict(list)
    n_edges = 0

    for global_in, amp_in in amplitudes.items():
        local_in = sum(
            ((int(global_in) >> int(q)) & 1) << local_slot
            for local_slot, q in enumerate(physical_qubits)
        )
        for local_out, matrix_element in enumerate(local_u[:, local_in]):
            if abs(matrix_element) <= PATH_MATRIX_ELEMENT_TOL:
                continue
            global_out = replace_local_bits(global_in, physical_qubits, local_out)
            contribution = complex(matrix_element) * complex(amp_in)
            next_amplitudes[global_out] += contribution
            next_counts[global_out] += int(path_counts[global_in])
            contributions[global_out].append(contribution)
            n_edges += 1

    # Elimina somente resíduos numéricos após a soma coerente.
    next_amplitudes = {
        int(index): complex(amp)
        for index, amp in next_amplitudes.items()
        if abs(amp) > PATH_AMPLITUDE_TOL
    }
    next_counts = {index: int(next_counts[index]) for index in next_amplitudes}

    interference_by_output = {}
    for out_index, terms in contributions.items():
        coherent = float(abs(sum(terms)) ** 2)
        incoherent = float(sum(abs(term) ** 2 for term in terms))
        interference_by_output[int(out_index)] = coherent - incoherent

    target_terms = contributions.get(PATH_TARGET_INDEX, [])
    target_coherent = float(abs(sum(target_terms)) ** 2) if target_terms else 0.0
    target_incoherent = float(sum(abs(term) ** 2 for term in target_terms))

    return next_amplitudes, next_counts, {
        "n_transition_edges": int(n_edges),
        "target_incoming_terms": int(len(target_terms)),
        "target_interference": float(target_coherent - target_incoherent),
        "interference_l1_all_outputs": float(sum(abs(v) for v in interference_by_output.values())),
        "constructive_interference_all_outputs": float(sum(max(v, 0.0) for v in interference_by_output.values())),
        "destructive_interference_all_outputs": float(-sum(min(v, 0.0) for v in interference_by_output.values())),
    }


initial_index = int(label_to_index[initial_bitstring_qiskit])
path_amplitudes = {initial_index: 1.0 + 0.0j}
path_counts = {initial_index: 1}
pathsum_rows = []

for step, theta_index in enumerate(ORIGINAL_BLOCK_ORDER, start=1):
    p_target_before = float(abs(path_amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
    path_amplitudes, path_counts, interference = pathsum_step(
        path_amplitudes, path_counts, int(theta_index), ACTION_THETA[int(theta_index)]
    )
    p_target_after = float(abs(path_amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
    total_norm = float(sum(abs(amp) ** 2 for amp in path_amplitudes.values()))

    pathsum_rows.append({
        "step": int(step),
        "theta_index": int(theta_index),
        "is_original7": bool(theta_index in ACTIVE_THETA_INDICES),
        "n_supported_basis_states": int(len(path_amplitudes)),
        "n_histories_to_target": int(path_counts.get(PATH_TARGET_INDEX, 0)),
        "p_target_before": p_target_before,
        "p_target_after": p_target_after,
        "delta_p_target": float(p_target_after - p_target_before),
        "norm_after_step": total_norm,
        **interference,
    })

pathsum_df = pd.DataFrame(pathsum_rows)

# AUDITORIA: a soma sobre caminhos deve reconstruir exatamente o Statevector final.
pathsum_state = np.zeros(2 ** N_ASSETS, dtype=np.complex128)
for basis_index, amplitude in path_amplitudes.items():
    pathsum_state[int(basis_index)] = complex(amplitude)
pathsum_fidelity = float(abs(np.vdot(ACTION_STATE, pathsum_state)) ** 2)
pathsum_norm_error = float(abs(np.vdot(pathsum_state, pathsum_state).real - 1.0))

if (1.0 - pathsum_fidelity) > 1e-10 or pathsum_norm_error > 1e-10:
    raise RuntimeError(
        f"Path-sum não reproduziu o circuito: F={pathsum_fidelity}, erro_norma={pathsum_norm_error}"
    )

print(f"Fidelidade path-sum/circuito = {pathsum_fidelity:.16f}")
print("Número de histórias não nulas chegando ao alvo =", path_counts.get(PATH_TARGET_INDEX, 0))
display(pathsum_df)
pathsum_df.to_csv(ACTION_TABLE_DIR / "discrete_pathsum_interference_by_block.csv", index=False)


### Célula 35 — onde a amplitude e a interferência aparecem

Os dois gráficos abaixo usam a **ordem física dos blocos**.

O primeiro mostra a probabilidade do bitstring alvo depois de cada bloco. O segundo mostra a magnitude de interferência gerada na distribuição naquele bloco.

As linhas verticais marcam apenas as posições ocupadas pelos 7 θ identificados na varredura anterior. Elas não entram no cálculo do path-sum.


In [ ]:
# ============================================================
# 35. VISUALIZAÇÃO DO CAMINHO DE AMPLITUDE E INTERFERÊNCIA
# ============================================================

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(pathsum_df["step"], pathsum_df["p_target_after"], marker="o", linewidth=1.5)
for row in pathsum_df.itertuples(index=False):
    if row.is_original7:
        ax.axvline(row.step, linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("bloco aplicado")
ax.set_ylabel(f"P({PATH_TARGET_BITSTRING})")
ax.set_title("Construção da probabilidade do bitstring ótimo ao longo dos 30 blocos")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "target_probability_along_blocks.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    pathsum_df["step"],
    pathsum_df["interference_l1_all_outputs"],
    marker="o", linewidth=1.5,
)
for row in pathsum_df.itertuples(index=False):
    if row.is_original7:
        ax.axvline(row.step, linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("bloco aplicado")
ax.set_ylabel("soma |interferência| nos estados de saída")
ax.set_title("Interferência coerente criada por cada bloco")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "interference_magnitude_along_blocks.png", dpi=180)
plt.show()


### Célula 36 — comprimento Fubini–Study de uma trajetória parametrizada

Para aproximar a ideia de uma “trajetória” no espaço variacional sem introduzir uma dinâmica física artificial, usamos a interpolação periódica mais curta entre:

- o vetor original da âncora;
- o vetor reotimizado da arquitetura original.

Para

\[
\theta(s)=\theta_{\rm início}+s\,\Delta\theta,\qquad 0\le s\le1,
\]

o comprimento induzido pela métrica é

\[
L=\int_0^1\sqrt{\dot\theta^Tg(\theta)\dot\theta}\,ds.
\]

Isso é um **comprimento da trajetória escolhida**, não a prova de que ela seja uma geodésica nem uma ação física.

Além disso, construímos um endpoint que move apenas os 7 θ desde a âncora até seus valores reotimizados e medimos sua fidelidade com o estado ótimo completo. Se essa fidelidade for próxima de 1, temos uma evidência direta de que os 23 deslocamentos restantes não são necessários para alcançar o mesmo estado físico naquela região.


In [ ]:
# ============================================================
# 36. TRAJETÓRIA PARAMETRIZADA NA MÉTRICA DE FUBINI–STUDY
# ============================================================

period_vector = parameter_map_df.sort_values("theta_index")["angular_period"].to_numpy(dtype=float)
theta_path_start = np.asarray(anchors_df.iloc[0]["theta_vector"], dtype=float).copy()

# Usa o menor deslocamento angular permitido por cada periodicidade.
theta_short_delta = np.asarray([
    shortest_delta_to_target(theta_path_start[j], ACTION_THETA[j], period_vector[j])
    for j in range(N_PARAMETERS)
], dtype=float)

# Endpoint que move SOMENTE os 7 parâmetros empiricamente sensíveis.
theta_active_only_endpoint = theta_path_start.copy()
theta_active_only_endpoint[ACTIVE_THETA_INDICES] += theta_short_delta[ACTIVE_THETA_INDICES]
active_only_state = structural_state(theta_active_only_endpoint)
active_only_endpoint_fidelity = float(abs(np.vdot(ACTION_STATE, active_only_state)) ** 2)


def qgt_metric_at_theta(theta_values):
    """QGT por diferenças centrais; usado somente nos poucos pontos da trajetória."""
    psi0 = structural_state(theta_values)
    derivatives = []
    for j in range(N_PARAMETERS):
        plus, minus = np.array(theta_values, float), np.array(theta_values, float)
        plus[j] += FD_STEP
        minus[j] -= FD_STEP
        derivatives.append((structural_state(plus) - structural_state(minus)) / (2.0 * FD_STEP))
    derivatives = np.asarray(derivatives, dtype=np.complex128)
    gram_local = derivatives.conj() @ derivatives.T
    overlap_local = derivatives.conj() @ psi0
    metric = np.real(gram_local - np.outer(overlap_local, np.conj(overlap_local)))
    return 0.5 * (metric + metric.T)


parameter_path_rows = []
if RUN_PARAMETER_PATH_GEOMETRY:
    s_grid = np.linspace(0.0, 1.0, PARAMETER_PATH_POINTS)
    for s_value in s_grid:
        theta_s = theta_path_start + float(s_value) * theta_short_delta
        g_s = qgt_metric_at_theta(theta_s)
        speed_sq = float(theta_short_delta @ g_s @ theta_short_delta)
        eig_s = np.clip(np.linalg.eigvalsh(g_s), 0.0, None)
        max_eig_s = float(eig_s.max()) if len(eig_s) else 0.0
        rank_s = int(np.sum(eig_s > max(1e-14, QGT_RELATIVE_NULL_TOL * max_eig_s)))
        diag_s = np.clip(np.diag(g_s), 0.0, None)
        trace_s = float(diag_s.sum())
        active_fraction_s = float(diag_s[active_mask].sum() / trace_s) if trace_s > 0 else np.nan
        parameter_path_rows.append({
            "s": float(s_value),
            "fubini_study_speed": float(np.sqrt(max(speed_sq, 0.0))),
            "qgt_numeric_rank": rank_s,
            "active7_metric_trace_fraction": active_fraction_s,
        })

    parameter_path_df = pd.DataFrame(parameter_path_rows)
    geometric_path_length = float(np.trapz(
        parameter_path_df["fubini_study_speed"].to_numpy(dtype=float),
        parameter_path_df["s"].to_numpy(dtype=float),
    ))
    parameter_path_df.to_csv(ACTION_TABLE_DIR / "parameter_interpolation_geometry.csv", index=False)
else:
    parameter_path_df = pd.DataFrame()
    geometric_path_length = np.nan

path_geometry_summary_df = pd.DataFrame([{
    "active_only_endpoint_fidelity_to_full_optimum": active_only_endpoint_fidelity,
    "full_interpolation_fubini_study_length": geometric_path_length,
    "parameter_path_points": int(PARAMETER_PATH_POINTS if RUN_PARAMETER_PATH_GEOMETRY else 0),
}])

display(path_geometry_summary_df)
if not parameter_path_df.empty:
    display(parameter_path_df)
path_geometry_summary_df.to_csv(ACTION_TABLE_DIR / "parameter_path_geometry_summary.csv", index=False)


### Célula 37 — tabela integrada: os mesmos θ aparecem em diagnósticos independentes?

A última tabela não cria um score arbitrário. Ela coloca lado a lado, para cada bloco:

- sensibilidade por perturbação do 20.15;
- \(g_{ii}\) do QGT;
- curvatura de energia;
- curvatura de \(P(\mathcal X_{opt})\);
- derivada da amplitude do ótimo;
- alteração de \(P(x_f)\) quando o bloco é aplicado;
- magnitude de interferência criada pelo bloco.

A pergunta é simples: **os mesmos sete índices aparecem repetidamente em quantidades que foram calculadas por mecanismos diferentes?**

Se sim, a interpretação de uma subvariedade ativa ganha força. Se os conjuntos divergirem, a divergência é o resultado científico — por exemplo, um θ pode ser geometricamente ativo, mas energeticamente plano.


In [ ]:
# ============================================================
# 37. EVIDÊNCIA INTEGRADA — SEM SCORE ARBITRÁRIO
# ============================================================

original_probe_df = sensitivity_by_architecture_df.loc[
    sensitivity_by_architecture_df["architecture"].eq("original")
, [
    "theta_index", "sensitive_after_reorder", "max_abs_delta_p_optimal",
    "max_abs_delta_energy", "max_tvd", "dominant_bitstring_changed",
]].copy()

pathsum_by_theta_df = pathsum_df[[
    "theta_index", "step", "delta_p_target", "target_interference",
    "interference_l1_all_outputs", "n_histories_to_target",
]].copy()

integrated_evidence_df = variational_geometry_df[[
    "theta_index", "ansatz_gate_type", "logical_block_qubits", "logical_assets",
    "originally_sensitive", "qgt_diag", "geometrically_null",
    "energy_gradient_fd", "energy_curvature_fd", "energetically_flat",
    "p_target_curvature_fd", "p_optimal_curvature_fd",
]].merge(
    amplitude_influence_df[[
        "theta_index", "abs_d_amplitude_target", "d_probability_target",
        "d_probability_all_optima", "amplitude_decoupled",
    ]], on="theta_index", how="left", validate="one_to_one"
).merge(
    original_probe_df, on="theta_index", how="left", validate="one_to_one"
).merge(
    pathsum_by_theta_df, on="theta_index", how="left", validate="one_to_one"
)

# Colunas normalizadas são apenas para comparação visual; nenhum limiar científico
# depende delas e nenhum "score final" é construído.
for source_column, normalized_column in [
    ("qgt_diag", "qgt_diag_relative"),
    ("energy_curvature_fd", "abs_energy_curvature_relative"),
    ("p_optimal_curvature_fd", "abs_p_optimal_curvature_relative"),
    ("abs_d_amplitude_target", "abs_d_amplitude_relative"),
    ("interference_l1_all_outputs", "interference_l1_relative"),
]:
    values = integrated_evidence_df[source_column].abs().to_numpy(dtype=float)
    scale = float(np.nanmax(values)) if np.any(np.isfinite(values)) else 0.0
    integrated_evidence_df[normalized_column] = values / scale if scale > 0 else 0.0

integrated_evidence_df = integrated_evidence_df.sort_values("step").reset_index(drop=True)
integrated_evidence_df.to_csv(
    ACTION_TABLE_DIR / "integrated_30theta_structural_geometric_pathsum_evidence.csv",
    index=False,
)

display(integrated_evidence_df[[
    "theta_index", "step", "originally_sensitive", "sensitive_after_reorder",
    "qgt_diag", "geometrically_null", "energy_curvature_fd", "energetically_flat",
    "p_optimal_curvature_fd", "abs_d_amplitude_target", "delta_p_target",
    "target_interference", "interference_l1_all_outputs",
]])

print("7 theta originalmente identificados:", ACTIVE_THETA_INDICES)
print(
    "theta não nulos geometricamente:",
    integrated_evidence_df.loc[~integrated_evidence_df["geometrically_null"], "theta_index"].astype(int).tolist(),
)
print(
    "theta sensíveis no probe estrutural original:",
    integrated_evidence_df.loc[integrated_evidence_df["sensitive_after_reorder"], "theta_index"].astype(int).tolist(),
)


## Checklist de auditoria antes de qualquer interpretação física

1. **Reconstrução por blocos:** a fidelidade entre o ansatz original e a reconstrução deve ser \(\approx1\). Se falhar, os testes de posição e o path-sum não devem ser usados.
2. **Controle de reotimização:** a arquitetura `original` precisa recuperar a solução já conhecida antes de comparar `active_first`, `active_last` ou `active_reverse_in_place`.
3. **Gradiente no mínimo:** `energy_gradient_fd` deve ser pequeno de forma geral. Isso é esperado; não use o gradiente para escolher os sete. Compare principalmente `qgt_diag` e `energy_curvature_fd`.
4. **QGT:** não conclua “dimensão 7” só porque sete diagonais são grandes. Verifique o **espectro completo**, `qgt_numeric_rank` e `qgt_participation_dimension`.
5. **Propagação por blocos:** `layer_final_fidelity` deve ser \(\approx1\).
6. **Path-sum:** `pathsum_fidelity` deve ser \(\approx1\) e o erro de norma deve ser desprezível. Essa é a certificação de que a soma discreta reproduz o circuito.
7. **Interferência:** `target_interference` mede interferência das contribuições que entram especificamente no bitstring alvo naquela camada; `interference_l1_all_outputs` mede a redistribuição de interferência em toda a base. Não são a mesma quantidade.
8. **Trajetória geométrica:** `full_interpolation_fubini_study_length` é o comprimento da interpolação escolhida, **não uma geodésica calculada** e não uma ação física.
9. **Subvariedade ativa:** a afirmação forte só fica sustentada se houver convergência de evidências independentes — perturbação, QGT, curvatura, amplitude, ordem dos blocos e path-sum — e não apenas coincidência com a lista `[2, 14, 17, 19, 22, 25, 27]`.

### Hipóteses que o 20.16 consegue separar

\[
\text{23 θ aparentemente inativos}
\;\Rightarrow\;
\begin{cases}
\text{direções geometricamente nulas},\\
\text{direções que mudam o estado mas são energeticamente planas},\\
\text{blocos fora do caminho de amplitude do ótimo},\\
\text{redundância dependente da posição/ordem},\\
\text{ou combinação desses mecanismos.}
\end{cases}
\]

Se os mesmos sete parâmetros dominarem a métrica de Fubini–Study, a rigidez energética e a construção/interferência de amplitude, então passa a existir evidência concreta de uma **subvariedade variacional efetiva de baixa dimensão**. O número dessa dimensão deve ser reportado a partir do espectro do QGT, não imposto previamente como sete.
